# 02 · Soup training, resource analysis and artifact verification on one T4

**Input:** Kaggle dataset `pernavjain/tool-calling` with exactly
`train.jsonl`, `validation.jsonl`, `test.jsonl`, `soup-data.yaml`.
**Base:** pinned `Qwen/Qwen2.5-1.5B-Instruct`. **Soup:** pinned `0.77.2` source commit.

Attach that dataset with **Add Input**, enable **Internet**, and select
**GPU T4 x2**. The notebook exposes only the first T4 to each worker.
Run cells in order in a fresh Kaggle session. All model-owning work runs
in separate Python processes; the notebook kernel does not load a model.

| Phase | What this notebook executes |
|---|---|
| 3 | Prepared-data identity/mask checks, Soup pre-flight, controlled failures and a scratch benchmark |
| 4 | Two epochs of rank8 LoRA for seeds42/43/44, best-epoch reload and validation-only selection |
| 5 | Clean adapter, dense merge, saved NF4, no-op quantized-base control and cold standalone reloads |
| 6 | B/A/M/Q comparisons, strict call scoring, grouped uncertainty, general regressions and a frozen-policy verdict |

The full default experiment includes three training runs and substantial
inference. Runtime depends on the allocated T4 and generated lengths;
the tiny smoke timing is not an estimate for this full run. Use the stage
logs and your Kaggle quota to monitor it. No external tools/APIs are executed.

B = base; A = base plus adapter; M = saved dense merge; Q = saved NF4;
Q0 = NF4 of the base using an initialized zero-effect adapter.
A successful process is not automatically a SHIP result. Missing evidence,
unresolved controls, insufficient gain or regressions yield DON'T SHIP.

Only the four dataset files are inputs. `artifacts/` and `evidence/` are
generated outputs. The notebook writes its embedded **standalone
`verify_artifacts.py`** into the run directory; no helper upload is needed.

## 1. Configure this immutable experiment

Leave the prepared split and masking settings unchanged. Defaults match
`IMPLEMENTATION.md`: batch1, accumulation8, FP16 on T4, two epochs, learning
rate1e-4, 5% warmup and three fixed training seeds. Each seed exports the
checkpoint with the lowest epoch validation loss; seed selection then uses
validation function-call accuracy. Architecture/LoRA arithmetic and the original memory
envelope are saved before any optimization. The final prepared length is
read from `soup-data.yaml` (currently1024); fit is tested by the benchmark.

Do not rerun this setup cell after an interrupted stage: it creates a new
run. Rerun the failed stage cell in the existing session. A completed stage
is reused only if its code, settings and recorded output hashes still match.
Partial training/export outputs require inspection or a fresh run; they are
never silently resumed as a pristine experiment.

**Updating an older notebook:** start a fresh session/run for this recipe.
Keep any earlier saved outputs. Overwriting an old run's runner or YAML
would invalidate its frozen configuration and benchmark evidence.

In [1]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import subprocess
import sys
import time

assert (3,11) <= sys.version_info[:2] < (3,14), "Soup requires Python3.11–3.13"
DATASET_DIR = None  # Set an explicit Kaggle input folder only if auto-detection is ambiguous.
SEEDS = [42,43,44]
REFLECTION = ""  # Optional initial note; add your final reflection to REPORT.md after the run.
STAGE_TIMEOUT_SECONDS = 4 * 60 * 60
candidates = [Path(DATASET_DIR)] if DATASET_DIR else [p.parent for p in Path('/kaggle/input').rglob('soup-data.yaml')
    if all((p.parent/name).is_file() for name in ('train.jsonl','validation.jsonl','test.jsonl'))]
assert len(candidates) == 1, "Attach pernavjain/tool-calling with Add Input, or set DATASET_DIR explicitly."
dataset_dir = candidates[0].resolve()
devices = subprocess.check_output(['nvidia-smi','--query-gpu=index,uuid,name','--format=csv,noheader'],text=True)
first = [part.strip() for part in devices.splitlines()[0].split(',')]
assert first[0] == '0' and 'T4' in first[2], "Select the Kaggle T4 pool."
os.environ['CUDA_VISIBLE_DEVICES'] = first[1]
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN = Path('/kaggle/working/soup-experiment') / RUN_ID
RUN.mkdir(parents=True,exist_ok=False)
(RUN/'evidence/logs').mkdir(parents=True)
settings = {'run_id':RUN_ID,'dataset_dir':str(dataset_dir),'seeds':SEEDS,
            'gpu_uuid':first[1],'physical_gpu_index':0,'reflection':REFLECTION}
(RUN/'settings.json').write_text(json.dumps(settings,indent=2)+'\n')
with (RUN/'evidence/nvidia-smi-initial.raw.log').open('w') as handle:
    subprocess.run(['nvidia-smi'],stdout=handle,stderr=subprocess.STDOUT,check=True)
print('Run:',RUN)
print('Dataset:',dataset_dir)
print('Selected physical GPU:',first)

Run: /kaggle/working/soup-experiment/20261110T044014005029Z
Dataset: /kaggle/input/datasets/pernavjain/tool-calling
Selected physical GPU: ['0', 'GPU-1abb4f06-438e-a094-a927-1e023c093229', 'Tesla T4']


## 2. Install the tested stack

The Soup Git commit and major training libraries are pinned. PyTorch is
retained from the Kaggle image when it satisfies Soup's requirements;
its exact version/CUDA build and all dependency diagnostics are saved.
The unused TorchAO backend is removed before importing PEFT, addressing
the observed Kaggle TorchAO0.10 incompatibility. The experiment uses
ordinary dense LoRA and bitsandbytes NF4, not TorchAO.

Full output goes directly to disk, preventing the bulk-console stall from
the earlier smoke run. `pip check` records shared-image conflicts; Soup's
own installed dependency-bound check is evaluated separately.

In [2]:
def external(argv,label,allowed=(0,),timeout=900):
    stamp = str(time.time_ns())
    log = RUN/f'evidence/logs/{label}-{stamp}.log'
    started = dt.datetime.now(dt.timezone.utc).isoformat()
    exit_code = None
    timed_out = False
    print(f'{label} started. Log: {log.relative_to(RUN)}',flush=True)
    try:
        with log.open('w') as handle:
            result = subprocess.run(argv,cwd=RUN,stdout=handle,stderr=subprocess.STDOUT,
                                    timeout=timeout,check=False)
        exit_code = result.returncode
    except subprocess.TimeoutExpired:
        timed_out = True
        raise
    finally:
        with (RUN/'evidence/commands.jsonl').open('a') as handle:
            handle.write(json.dumps({'argv':argv,'label':label,'started_utc':started,
                'finished_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
                'exit_code':exit_code,'timed_out':timed_out,'log':str(log.relative_to(RUN))})+'\n')
        if timed_out or (exit_code is not None and exit_code not in allowed):
            tail = '\n'.join(log.read_text(errors='replace').splitlines()[-60:])[-8000:]
            failure = {'label':label,'argv':argv,'exit_code':exit_code,'timed_out':timed_out,
                       'log':str(log.relative_to(RUN)),'tail':tail}
            (RUN/'evidence/last-failure.json').write_text(json.dumps(failure,indent=2)+'\n')
    print(f'{label}: exit={result.returncode}; log={log.relative_to(RUN)}')
    if result.returncode not in allowed:
        raise RuntimeError(f'{label} failed (exit {result.returncode})\nLog: {log}\n'
                           f'Diagnostic record: {RUN / "evidence/last-failure.json"}\n{tail}')
    return result.returncode

external([sys.executable,'-m','pip','install','--disable-pip-version-check',
    'soup-cli[train] @ git+https://github.com/MakazhanAlpamys/Soup.git@4cff43bf967e96a484381205e5125e2fc39227f1',
    'transformers==5.20.2','peft==0.21.2','trl==0.29.1','bitsandbytes==0.50.2','accelerate==1.15.0',
    'jsonschema[format-nongpl]==4.26.0','jinja2>=3.1','psutil>=5.9','reportlab>=4,<5','pypdf>=6,<7'],
    'install',timeout=1800)
external([sys.executable,'-m','pip','uninstall','-y','torchao'],'remove-optional-torchao')
external([sys.executable,'-m','pip','check'],'pip-check',allowed=(0,1))
external([sys.executable,'-m','pip','freeze'],'pip-freeze')

install started. Log: evidence/logs/install-1794285614407561267.log
install: exit=0; log=evidence/logs/install-1794285614407561267.log
remove-optional-torchao started. Log: evidence/logs/remove-optional-torchao-1794285632114856137.log
remove-optional-torchao: exit=0; log=evidence/logs/remove-optional-torchao-1794285632114856137.log
pip-check started. Log: evidence/logs/pip-check-1794285632881839149.log
pip-check: exit=1; log=evidence/logs/pip-check-1794285632881839149.log
pip-freeze started. Log: evidence/logs/pip-freeze-1794285634901677652.log
pip-freeze: exit=0; log=evidence/logs/pip-freeze-1794285634901677652.log


0

## 3. Materialize the standalone runner/verifier

This cell only writes the embedded source. Each following stage launches
it in a fresh process. Inspect `verify_artifacts.py` in the Output panel
for the implementation and command-line interface. It can evaluate the
retained artifacts later without rerunning training or this notebook.

In [3]:
RUNTIME_SOURCE = '"""Standalone Soup experiment runner and verifier; embedded in the Kaggle notebook.\n\nOnly the standard library is imported at module load so pure checks need no GPU.\nRun --help for stage commands. Reports describe observed results, never intentions.\n"""\n\nimport argparse\nimport collections\nimport contextlib\nimport copy\nimport csv\nimport datetime as dt\nfrom decimal import Decimal\nimport faulthandler\nimport functools\nimport hashlib\nimport importlib.metadata as metadata\nimport inspect\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport re\nimport shutil\nimport statistics\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport unicodedata\n\nSOUP_REV = "4cff43bf967e96a484381205e5125e2fc39227f1"\nMODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"\nMODEL_REV = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"\nT4_ALLOCATOR_CONFIG = "backend:native,expandable_segments:True,garbage_collection_threshold:0.80"\nEXPECTED_DATA = {\n    "train.jsonl": "3e1c19821c79e777c6e1524e3b94880c6ecda03240f630f25b8c945b31e418bf",\n    "validation.jsonl": "d46eee968314585345c6fd50f570ddfdde242f226f828d22b0ad1d32239063da",\n    "test.jsonl": "b7f4c4942fc77e9bf16ab0812be382de458a2077b0eed846824e81b055a7b19f",\n    "soup-data.yaml": "1daab66b6b161b65fc9f2dfe6b63cfd0288750132b8627b5508ac4a67f1fd9f8",\n}\nROOT = None\nSETTINGS = None\nTORCH = None\nCAPTURE = None\nTRAINER_POLICY = {\n    "eval_strategy": "epoch", "save_strategy": "epoch", "do_eval": True,\n    "load_best_model_at_end": True, "metric_for_best_model": "eval_loss", "greater_is_better": False,\n    "save_total_limit": 2, "per_device_eval_batch_size": 1, "prediction_loss_only": True,\n    "gradient_checkpointing_kwargs": {"use_reentrant": False}, "dataloader_num_workers": 0,\n    "disable_tqdm": True,\n}\nREQUIRED_INTEGRITY = (\n    "required_stage_reports_pass", "policy_and_test_unchanged", "artifact_hashes_bound",\n    "updated_and_active_adapter", "selected_frozen_weights_unchanged", "disabled_base_agreement",\n    "merge_fidelity", "scorer_data_controls", "base_for_merge_control", "q0_for_q_control",\n    "Soup_noop_adapter_control", "portable_M_Q", "real_saved_nf4",\n    "no_new_critical_challenge_failures", "one_gib_tested_headroom", "three_seed_development_complete",\n    "Soup_environment_bounds_ok", "Soup_fixture_coverage_resolved",\n)\n\n\ndef now():\n    return dt.datetime.now(dt.timezone.utc).isoformat()\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)\n\n\ndef digest(value):\n    return hashlib.sha256(canonical(value).encode()).hexdigest()\n\n\ndef file_sha(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(2**20), b""):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef benchmark_headroom(report, allocator_config):\n    """Use the more conservative of device usage and allocator reservation."""\n    resources = report.get("resources", {})\n    hardware = report.get("hardware", {})\n    capacity = hardware.get("total_bytes", 0)\n    sampled = resources.get("peak_device_used_mib_sampled", 0) * 2**20\n    allocated = resources.get("peak_allocated_bytes", 0)\n    reserved = resources.get("peak_reserved_bytes", 0)\n    remaining = capacity - max(sampled, reserved)\n    measured = (\n        report.get("status") == "passed" and hardware.get("visible_count") == 1\n        and "T4" in hardware.get("name", "") and capacity > 0\n        and resources.get("device_samples", 0) > 0 and sampled > 0 and 0 < allocated <= reserved\n        and not any("cuda_measurement_error" in segment for segment in resources.get("segments", []))\n        and hardware.get("allocator_backend") == "native" and hardware.get("allocator_config") == allocator_config\n    )\n    return {\n        "passed": measured and remaining >= 2**30,\n        "measurement_complete": measured,\n        "usable_device_gib": round(capacity / 2**30, 3),\n        "sampled_device_peak_gib": round(sampled / 2**30, 3),\n        "allocated_peak_gib": round(allocated / 2**30, 3),\n        "reserved_peak_gib": round(reserved / 2**30, 3),\n        "headroom_gib": round(remaining / 2**30, 3) if measured else None,\n        "required_headroom_gib": 1,\n        "allocator_config": hardware.get("allocator_config"),\n    }\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + ".tmp")\n    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n")\n    temp.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\n\ndef write_jsonl(path, rows):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w") as handle:\n        for row in rows:\n            handle.write(canonical(row) + "\\n")\n\n\ndef read_jsonl(path):\n    return [strict_json(line) for line in Path(path).read_text().splitlines() if line.strip()]\n\n\ndef strict_json(text):\n    def pairs(items):\n        result = {}\n        for key, value in items:\n            if key in result:\n                raise ValueError("duplicate key: " + key)\n            result[key] = value\n        return result\n\n    def number(value):\n        result = float(value)\n        if not math.isfinite(result) or Decimal(value) != Decimal(str(result)):\n            raise ValueError("nonfinite or precision-losing number")\n        return result\n\n    def invalid(value):\n        raise ValueError("nonstandard JSON constant: " + value)\n\n    return json.loads(text, object_pairs_hook=pairs, parse_float=number, parse_constant=invalid)\n\n\ndef parse_calls(text):\n    if not isinstance(text, str):\n        raise ValueError("response is not text")\n    calls, position = [], 0\n    while position < len(text):\n        while position < len(text) and text[position].isspace():\n            position += 1\n        if position == len(text):\n            break\n        match = re.compile(r"<tool_call>\\s*(.*?)\\s*</tool_call>", re.S).match(text, position)\n        if match is None:\n            raise ValueError(f"unexpected text/incomplete tag at {position}")\n        call = strict_json(match.group(1))\n        if not isinstance(call, dict) or set(call) != {"name", "arguments"}:\n            raise ValueError("call keys must be name and arguments")\n        if not isinstance(call["name"], str) or not call["name"] or not isinstance(call["arguments"], dict):\n            raise ValueError("invalid call name/arguments")\n        calls.append(call)\n        position = match.end()\n    if not calls:\n        raise ValueError("no complete calls")\n    return calls\n\n\ndef tagged(calls):\n    return "\\n".join("<tool_call>\\n" + canonical(call) + "\\n</tool_call>" for call in calls)\n\n\ndef json_equal(a, b):\n    pending = [(a,b)]\n    while pending:\n        x,y = pending.pop()\n        if isinstance(x,bool) or isinstance(y,bool):\n            if type(x) is not type(y) or x != y:return False\n        elif isinstance(x,(int,float)) and isinstance(y,(int,float)):\n            if x != y:return False\n        elif type(x) is not type(y):\n            return False\n        elif isinstance(x,dict):\n            if x.keys() != y.keys():return False\n            pending.extend((x[k],y[k]) for k in x)\n        elif isinstance(x,list):\n            if len(x) != len(y):return False\n            pending.extend(zip(x,y))\n        elif x != y:\n            return False\n    return True\n\n\n@functools.lru_cache(maxsize=10000)\ndef schema_validator(schema_text):\n    from jsonschema import Draft7Validator, FormatChecker, validators\n    schema = strict_json(schema_text)\n    checker = FormatChecker()\n\n    def guard(value):\n        if isinstance(value, dict):\n            if any(key in value for key in ("$dynamicRef", "$recursiveRef", "nullable")):\n                raise ValueError("unsupported schema keyword")\n            if "$ref" in value and not str(value["$ref"]).startswith("#"):\n                raise ValueError("external schema references forbidden")\n            if "format" in value and value["format"] not in checker.checkers:\n                raise ValueError("unknown schema format")\n            for item in value.values():\n                guard(item)\n        elif isinstance(value, list):\n            for item in value:\n                guard(item)\n    guard(schema)\n    if schema.get("type") != "object":\n        raise ValueError("parameters must declare object type")\n    known = {"http://json-schema.org/draft-04/schema#", "http://json-schema.org/draft-06/schema#",\n             "http://json-schema.org/draft-07/schema#", "https://json-schema.org/draft/2019-09/schema",\n             "https://json-schema.org/draft/2020-12/schema"}\n    if "$schema" in schema and schema["$schema"] not in known:\n        raise ValueError("unsupported schema dialect")\n    cls = validators.validator_for(schema, default=Draft7Validator)\n    cls.check_schema(schema)\n    return cls(schema, format_checker=checker)\n\n\ndef functions_from_messages(messages):\n    if [m.get("role") for m in messages] != ["system", "user", "assistant"]:\n        raise ValueError("expected explicit system/user/assistant messages")\n    blocks = [x.strip() for x in re.findall(r"<tools>(.*?)</tools>", messages[0]["content"], flags=re.S) if x.strip()]\n    if len(blocks) != 1:\n        raise ValueError("expected one nonempty tools block")\n    tools = strict_json(blocks[0])\n    if not isinstance(tools, list) or not tools:\n        raise ValueError("empty tools")\n    result = {}\n    for tool in tools:\n        if tool.get("type") != "function":\n            raise ValueError("unsupported tool type")\n        function = tool["function"]\n        name = function["name"]\n        if not isinstance(name, str) or not name or name in result:\n            raise ValueError("missing/duplicate tool name")\n        schema_validator(canonical(function["parameters"]))\n        result[name] = function\n    return result\n\n\ndef score_turn(text, expected, functions, truncated=False, matcher=None):\n    result = {key: False for key in ("parse_ok", "known_tools", "tool_selection", "schema_ok", "arguments_ok", "complete")}\n    result.update(truncated=bool(truncated), extra_calls=False, unknown_calls=False)\n    try:\n        actual = parse_calls(text)\n    except (ValueError, TypeError, RecursionError) as exc:\n        return {**result, "error": str(exc)}\n    result["parse_ok"] = True\n    result["extra_calls"] = len(actual) > len(expected)\n    result["known_tools"] = all(call["name"] in functions for call in actual)\n    result["unknown_calls"] = not result["known_tools"]\n    result["tool_selection"] = [x["name"] for x in actual] == [x["name"] for x in expected]\n    schema_ok = result["known_tools"]\n    if schema_ok:\n        for call in actual:\n            try:\n                schema_validator(canonical(functions[call["name"]]["parameters"])).validate(call["arguments"])\n            except Exception as exc:\n                schema_ok = False\n                result["schema_error"] = str(exc)[:500]\n    result["schema_ok"] = schema_ok\n    result["arguments_ok"] = len(actual) == len(expected) and all(json_equal(x["arguments"], y["arguments"]) for x, y in zip(actual, expected))\n    matches = []\n    if len(actual) == len(expected):\n        for a, b in zip(actual, expected):\n            matches.append(bool(matcher(canonical({"function": a}), canonical({"function": b})))\n                           if matcher is not None else json_equal(a, b))\n    result["complete"] = bool(not truncated and schema_ok and len(actual) == len(expected) and matches and all(matches))\n    return result\n\n\ndef grouped_pairs(before, after, replicates=10000, seed=42):\n    if len(before) != len(after) or not before:\n        raise ValueError("paired denominators differ or are empty")\n    groups = collections.defaultdict(lambda: [0.0, 0])\n    for b, a in zip(before, after):\n        if b["id"] != a["id"] or b["group"] != a["group"]:\n            raise ValueError("paired IDs/groups do not match")\n        groups[b["group"]][0] += float(a["complete"]) - float(b["complete"])\n        groups[b["group"]][1] += 1\n    values = list(groups.values())\n    rng = random.Random(seed)\n    draws = []\n    for _ in range(replicates):\n        chosen = [values[rng.randrange(len(values))] for _ in values]\n        draws.append(sum(x[0] for x in chosen) / sum(x[1] for x in chosen))\n    draws.sort()\n    return {"delta": sum(x[0] for x in values) / len(before), "ci95": [draws[int(.025 * (replicates - 1))], draws[int(.975 * (replicates - 1))]],\n            "turns": len(before), "groups": len(values), "replicates": replicates, "method": "paired cluster bootstrap, all turns retained"}\n\n\ndef summarize_predictions(rows):\n    if not rows:\n        raise ValueError("no predictions")\n    keys = ("parse_ok", "known_tools", "tool_selection", "schema_ok", "arguments_ok", "complete", "truncated", "extra_calls", "unknown_calls")\n    result = {key: sum(bool(x[key]) for x in rows) / len(rows) for key in keys}\n    result["n"] = len(rows)\n    result["mean_assistant_nll"] = sum(x["nll_sum"] for x in rows) / sum(x["nll_tokens"] for x in rows)\n    result["slices"] = {}\n    for label, members in (("single_call", [x for x in rows if x["call_count"] == 1]),\n                           ("multiple_call", [x for x in rows if x["call_count"] > 1]),\n                           ("length_le_512", [x for x in rows if x["reference_tokens"] <= 512]),\n                           ("length_gt_512", [x for x in rows if x["reference_tokens"] > 512])):\n        result["slices"][label] = {"n": len(members), "complete": sum(x["complete"] for x in members) / len(members) if members else None}\n    result["latency_seconds"] = {"median": statistics.median(x["generation_seconds"] for x in rows),\n                                 "p95": sorted(x["generation_seconds"] for x in rows)[math.ceil(.95*len(rows))-1]}\n    return result\n\n\ndef decision_gates(pair_bq, pair_mq, general_b, general_q, integrity):\n    gates = {name:False for name in REQUIRED_INTEGRITY}\n    gates.update(integrity)\n    gates.update({\n        "task_gain_ge_5pp": round(pair_bq["delta"],12) >= .05,\n        "task_gain_ci_positive": pair_bq["ci95"][0] > 0,\n        "quant_point_loss_le_2pp": round(pair_mq["delta"],12) >= -.02,\n        "quant_ci_lower_ge_minus5pp": round(pair_mq["ci95"][0],12) >= -.05,\n        "general_suite_regression_le_5pp": bool(general_b) and general_b.keys() == general_q.keys()\n            and all(round(general_q[k] - general_b[k],12) >= -.05 for k in general_b),\n    })\n    return {"verdict": "SHIP" if all(value is True for value in gates.values()) else "DON\'T SHIP",\n            "gates": gates, "blockers": [name for name, passed in gates.items() if passed is not True]}\n\n\ndef event(kind, **values):\n    row = {"utc": now(), "event": kind, **values}\n    path = ROOT / "evidence/events.jsonl"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("a") as handle:\n        handle.write(canonical(row) + "\\n")\n    if kind in {"stage_start", "stage_end", "segment", "progress"}:\n        print(canonical(row), file=sys.stderr, flush=True)\n\n\nclass ResourceCapture:\n    def __init__(self, label):\n        import psutil\n        self.psutil = psutil\n        self.label = label\n        self.started = time.monotonic()\n        self.stop = threading.Event()\n        self.segments = []\n        self.peak_rss = 0\n        self.device_peak_mib = 0\n        self.device_samples = 0\n        self.phase_name = "startup"\n        self.thread = None\n\n    def sample(self):\n        process = self.psutil.Process()\n        raw_path = ROOT / "evidence/nvidia-smi.raw.log"\n        while not self.stop.is_set():\n            record = {"utc": now(), "stage": self.label, "phase": self.phase_name,\n                      "rss_bytes": process.memory_info().rss, "available_ram_bytes": self.psutil.virtual_memory().available,\n                      "disk_free_bytes": shutil.disk_usage(ROOT).free}\n            self.peak_rss = max(self.peak_rss, record["rss_bytes"])\n            try:\n                command = ["nvidia-smi", "--query-gpu=timestamp,index,uuid,pci.bus_id,name,memory.total,memory.used,utilization.gpu,clocks.sm,power.draw", "--format=csv"]\n                observed = subprocess.run(command, capture_output=True, text=True, timeout=10, check=False)\n                with raw_path.open("a") as handle:\n                    handle.write(f"# {canonical(record)} exit={observed.returncode}\\n{observed.stdout}{observed.stderr}")\n                if observed.returncode == 0:\n                    for row in csv.reader(observed.stdout.splitlines()[1:], skipinitialspace=True):\n                        if len(row) == 10 and row[1] == "0":\n                            self.device_peak_mib = max(self.device_peak_mib, int(row[6].split()[0]))\n                            self.device_samples += 1\n            except (OSError, subprocess.SubprocessError, ValueError) as exc:\n                record["sample_error"] = f"{type(exc).__name__}: {exc}"\n            with (ROOT / "evidence/host-samples.jsonl").open("a") as handle:\n                handle.write(canonical(record) + "\\n")\n            self.stop.wait(.2)\n\n    def mark(self, name, reset=True):\n        global TORCH\n        record = {"name": name, "utc": now(), "elapsed_seconds": time.monotonic() - self.started}\n        if TORCH is not None and TORCH.cuda.is_initialized():\n            try:\n                TORCH.cuda.synchronize()\n                record.update(allocated_bytes=TORCH.cuda.memory_allocated(), reserved_bytes=TORCH.cuda.memory_reserved(),\n                              peak_allocated_bytes=TORCH.cuda.max_memory_allocated(), peak_reserved_bytes=TORCH.cuda.max_memory_reserved())\n                if reset:\n                    TORCH.cuda.reset_peak_memory_stats()\n            except RuntimeError as exc:\n                record["cuda_measurement_error"] = str(exc)\n        self.segments.append(record)\n        self.phase_name = name\n        event("segment", stage=self.label, **record)\n        return record\n\n    @contextlib.contextmanager\n    def phase(self, name):\n        self.mark("before_" + name)\n        try:\n            yield\n        finally:\n            self.mark("after_" + name)\n\n    def start(self):\n        event("stage_start", stage=self.label)\n        self.thread = threading.Thread(target=self.sample, daemon=True)\n        self.thread.start()\n\n    def finish(self):\n        self.mark("stage_end")\n        self.stop.set()\n        self.thread.join(timeout=15)\n        return {"elapsed_seconds": time.monotonic() - self.started, "peak_rss_bytes_sampled": self.peak_rss,\n                "peak_device_used_mib_sampled": self.device_peak_mib, "device_samples": self.device_samples,\n                "segments": self.segments, "peak_allocated_bytes": max((x.get("peak_allocated_bytes", 0) for x in self.segments), default=0),\n                "peak_reserved_bytes": max((x.get("peak_reserved_bytes", 0) for x in self.segments), default=0),\n                "scope": "Child-local allocator segments; 0.2 s target device/RSS sampling may miss transients. Bench CLI resets its own allocator after setup."}\n\n\ndef cuda():\n    global TORCH\n    import torch\n    TORCH = torch\n    torch.set_num_threads(2)\n    assert torch.cuda.is_available() and torch.cuda.device_count() == 1, "Exactly one visible CUDA GPU is required"\n    torch.cuda.init()\n    assert "T4" in torch.cuda.get_device_name(0), "Use the Kaggle T4 pool"\n    CAPTURE.mark("empty_cuda_context")\n    return torch\n\n\ndef cfg(seed=42):\n    from soup_cli.config.loader import load_config\n    return load_config(str(ROOT / f"configs/seed-{seed}.yaml"))\n\n\ndef rows(split):\n    return read_jsonl(ROOT / f"data/{split}.jsonl")\n\n\ndef tokenizer(directory=None):\n    from transformers import AutoTokenizer\n    return AutoTokenizer.from_pretrained(directory or ROOT / "artifacts/base", local_files_only=True, trust_remote_code=False)\n\n\ndef formatter(tok):\n    from soup_cli.data.sft_format import build_format_row\n    config = cfg(SETTINGS["seeds"][0])\n    return build_format_row(tok, config.data, training_cfg=config.training)\n\n\ndef audit_mask(messages, tok, format_fn):\n    full = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=False, return_dict=False)\n    prompt = tok.apply_chat_template(messages[:-1], tokenize=True, add_generation_prompt=True, return_dict=False)\n    if len(full) > SETTINGS["max_length"]:\n        raise ValueError("complete reference exceeds frozen max_length")\n    if full[:len(prompt)] != prompt:\n        raise ValueError("inference prefix changed")\n    encoded = format_fn({"messages": messages})\n    eot = full.index(tok.eos_token_id, len(prompt))\n    labels = [-100] * len(prompt) + full[len(prompt):eot+1] + [-100] * (len(full)-eot-1)\n    if encoded["input_ids"] != full or encoded["labels"] != labels:\n        raise ValueError("exact response/EOT label policy violated")\n    preferred = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=False, return_dict=True, return_assistant_tokens_mask=True)\n    if preferred.get("assistant_masks") != [int(x != -100) for x in labels]:\n        raise ValueError("generation-mask path unavailable; heuristic fallback forbidden")\n    if encoded["attention_mask"] != [1] * len(full) or not any(x != -100 for x in labels[1:]):\n        raise ValueError("invalid attention/shifted labels")\n    if tok.decode([x for x in labels if x != -100], skip_special_tokens=False, clean_up_tokenization_spaces=False) != messages[-1]["content"] + tok.eos_token:\n        raise ValueError("target/EOT text changed")\n    return encoded\n\n\ndef model_manifest(path):\n    path = Path(path)\n    files = {p.name: {"sha256": file_sha(p), "bytes": p.stat().st_size} for p in sorted(path.iterdir())\n             if p.is_file() and p.suffix in {".json", ".safetensors", ".jinja", ".txt", ".model"} and p.name != "provenance.json"}\n    return {"path": str(path.relative_to(ROOT)), "files": files, "sha256": digest(files)}\n\n\ndef check_bound_artifact(binding, directory):\n    actual = model_manifest(directory)\n    return actual["sha256"] == binding["sha256"]\n\n\ndef run_soup(arguments, allowed=(0,)):\n    from soup_cli.cli import app\n    code = 0\n    try:\n        result = app(args=arguments, standalone_mode=False)\n        if type(result) is int:\n            code = result\n    except SystemExit as exc:\n        code = int(exc.code or 0)\n    except Exception as exc:\n        import typer\n        if isinstance(exc, typer.Exit):\n            code = exc.exit_code\n        else:\n            raise\n    if code not in allowed:\n        raise RuntimeError(f"Soup {arguments[:3]} exited {code}; allowed={allowed}")\n    return code\n\n\ndef base_config_mapping(fragment, seed):\n    return {"base": "./artifacts/base", "task": "sft", "backend": "transformers",\n            "data": {**fragment["data"], "train": "./data/train.jsonl", "val_split": 0},\n            "training": {"epochs": 2, "lr": .0001, "batch_size": 1, "gradient_accumulation_steps": 8,\n                "optimizer": "adamw_torch", "scheduler": "cosine", "warmup_ratio": .05, "weight_decay": .01,\n                "max_grad_norm": 1., "gradient_checkpointing": True, "quantization": "none", "packing": False,\n                "train_on_eot": True, "seed": seed, "data_seed": seed, "logging_steps": 1, "save_steps": 200,\n                "lora": {"r": 8, "alpha": 16, "dropout": .05,\n                         "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}},\n            "output": f"./artifacts/training-{seed}"}\n\n\ndef configure_training(trainer):\n    from transformers.trainer_callback import ProgressCallback\n    from transformers.trainer_utils import IntervalStrategy, SaveStrategy\n    for name, value in TRAINER_POLICY.items():\n        setattr(trainer.args, name, copy.deepcopy(value))\n    trainer.args.eval_strategy = IntervalStrategy(TRAINER_POLICY["eval_strategy"])\n    trainer.args.save_strategy = SaveStrategy(TRAINER_POLICY["save_strategy"])\n    trainer.model.config.use_cache = False\n    # The progress callback is installed during construction, before these overrides.\n    trainer.remove_callback(ProgressCallback)\n\n\ndef checkpoint_evidence(trainer, export_dir):\n    import numpy as np\n    from safetensors import safe_open\n    evaluations = [{"epoch": float(row["epoch"]), "step": int(row["step"]), "eval_loss": float(row["eval_loss"])}\n                   for row in trainer.state.log_history if "eval_loss" in row]\n    epochs = int(trainer.args.num_train_epochs)\n    if ({row["epoch"] for row in evaluations} != set(range(1, epochs + 1))\n        or not all(math.isfinite(row["eval_loss"]) for row in evaluations)\n        or not math.isclose(trainer.state.epoch, epochs)):\n        raise ValueError("Missing complete, finite validation history for each training epoch")\n    best = min(evaluations, key=lambda row: (row["eval_loss"], row["step"]))\n    if (trainer.state.best_global_step != best["step"]\n        or trainer.state.best_metric is None\n        or not math.isclose(trainer.state.best_metric, best["eval_loss"], rel_tol=1e-7)):\n        raise ValueError("Trainer best checkpoint disagrees with validation-loss selection")\n    checkpoint = Path(trainer.state.best_model_checkpoint or "").resolve()\n    expected = checkpoint / "adapter_model.safetensors"\n    exported = Path(export_dir) / "adapter_model.safetensors"\n    # Compare tensors: serialization metadata/order can change without changing weights.\n    with safe_open(expected, framework="np") as before, safe_open(exported, framework="np") as after:\n        if set(before.keys()) != set(after.keys()):\n            raise ValueError("Exported adapter keys differ from the best validation checkpoint")\n        values = 0\n        for name in before.keys():\n            left, right = before.get_tensor(name), after.get_tensor(name)\n            if left.dtype != right.dtype or not np.array_equal(left, right) or not np.isfinite(right).all():\n                raise ValueError("Exported adapter differs from the best validation checkpoint: " + name)\n            values += right.size\n        tensors = len(before.keys())\n    if tensors != 392 or values != 9232384:\n        raise ValueError("Unexpected exported LoRA tensor/weight count")\n    return {"metric": "eval_loss", "selection_uses_test": False, "epoch_validation": evaluations,\n            "epochs_completed": trainer.state.epoch, "optimizer_steps_completed": trainer.state.global_step,\n            "selected_epoch": best["epoch"], "selected_step": best["step"], "best_validation_loss": best["eval_loss"],\n            "checkpoint": str(checkpoint.relative_to(ROOT)), "checkpoint_adapter_sha256": file_sha(expected),\n            "export_matches_checkpoint_tensors": True, "trainable_tensor_count": tensors, "trainable_parameter_count": values}\n\n\ndef group_metadata(all_rows):\n    parent = list(range(len(all_rows)))\n    owners = {}\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n    for i, row in enumerate(all_rows):\n        keys = [("request", " ".join(unicodedata.normalize("NFKC", row["messages"][1]["content"]).casefold().split()))]\n        keys += [("tool", name.casefold()) for name in row["functions"]]\n        for key in keys:\n            if key in owners:\n                a, b = find(i), find(owners[key])\n                parent[max(a, b)] = min(a, b)\n            else:\n                owners[key] = i\n    components = collections.defaultdict(list)\n    for i, row in enumerate(all_rows):\n        components[find(i)].append(row)\n    for members in components.values():\n        if len({row["split"] for row in members}) != 1:\n            raise ValueError("cross-split tool/request family overlap")\n        group = digest(sorted(row["id"] for row in members))\n        for row in members:\n            row["group"] = group\n    return all_rows\n\n\ndef challenge_rows():\n    tools = [\n        {"type": "function", "function": {"name": "check_city_weather", "parameters": {"type": "object", "properties": {"city": {"type": "string"}, "unit": {"enum": ["celsius", "fahrenheit"]}}, "required": ["city", "unit"], "additionalProperties": False}}},\n        {"type": "function", "function": {"name": "set_alert_enabled", "parameters": {"type": "object", "properties": {"enabled": {"type": "boolean"}}, "required": ["enabled"], "additionalProperties": False}}},\n        {"type": "function", "function": {"name": "multiply_values", "parameters": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"], "additionalProperties": False}}},\n    ]\n    cases = [("Get the weather for Paris in celsius.", [{"name": "check_city_weather", "arguments": {"city": "Paris", "unit": "celsius"}}]),\n             ("Get the weather for Tokyo in fahrenheit.", [{"name": "check_city_weather", "arguments": {"city": "Tokyo", "unit": "fahrenheit"}}]),\n             ("Enable the alert.", [{"name": "set_alert_enabled", "arguments": {"enabled": True}}]),\n             ("Disable the alert.", [{"name": "set_alert_enabled", "arguments": {"enabled": False}}]),\n             ("Multiply 0 by 7.", [{"name": "multiply_values", "arguments": {"a": 0, "b": 7}}]),\n             ("First get Paris weather in celsius, then disable the alert.", [\n                 {"name": "check_city_weather", "arguments": {"city": "Paris", "unit": "celsius"}},\n                 {"name": "set_alert_enabled", "arguments": {"enabled": False}}])]\n    system = "Return only the requested function calls, in order, within <tool_call> tags. Available tools: <tools>" + canonical(tools) + "</tools>"\n    return [{"messages": [{"role": "system", "content": system}, {"role": "user", "content": query},\n                          {"role": "assistant", "content": tagged(expected)}]} for query, expected in cases]\n\n\ndef prepare_stage():\n    import yaml\n    from huggingface_hub import snapshot_download\n    from transformers import AutoTokenizer\n    from soup_cli.config.loader import load_config_from_string\n    direct_url = json.loads(metadata.distribution("soup-cli").read_text("direct_url.json") or "{}")\n    if direct_url.get("vcs_info",{}).get("commit_id") != SOUP_REV:\n        raise ValueError("Installed Soup source commit does not match")\n    environment = {"python":sys.version,"versions":{name:metadata.version(name) for name in\n        ("soup-cli","torch","transformers","peft","trl","bitsandbytes","accelerate","jsonschema")},\n        "soup_install":direct_url,"runtime_sha256":file_sha(__file__)}\n    write_json(ROOT/"evidence/environment.json",environment)\n    source = Path(SETTINGS["dataset_dir"])\n    for name, sha in EXPECTED_DATA.items():\n        if file_sha(source / name) != sha:\n            raise ValueError(f"Dataset identity changed: {name}; freeze/review a new dataset version explicitly")\n        target = ROOT / ("data/" + name if name.endswith("jsonl") else "soup-data.yaml")\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(source / name, target)\n    fragment = yaml.safe_load((ROOT / "soup-data.yaml").read_text())\n    if fragment["data"]["val_split"] != 0 or not fragment["data"]["train_on_responses_only"]:\n        raise ValueError("Prepared split/mask settings changed")\n    SETTINGS["max_length"] = fragment["data"]["max_length"]\n    write_json(ROOT / "settings.json", SETTINGS)\n    base = ROOT / "artifacts/base"\n    snapshot_download(MODEL_ID, revision=MODEL_REV, local_dir=str(base),\n                      allow_patterns=["*.safetensors", "*.json", "*.txt", "*.model", "*.jinja", "README.md", "LICENSE"])\n    tok = AutoTokenizer.from_pretrained(base, local_files_only=True, trust_remote_code=False)\n    native = tok.chat_template\n    (ROOT / "evidence/native-template.jinja").write_text(native)\n    tok.chat_template = fragment["data"]["chat_template"]\n    tok.padding_side = "right"\n    tok.save_pretrained(base)\n    if not (ROOT / "configs").exists():\n        (ROOT / "configs").mkdir()\n    for seed in SETTINGS["seeds"]:\n        mapping = base_config_mapping(fragment, seed)\n        text = yaml.safe_dump(mapping, sort_keys=False)\n        load_config_from_string(text)\n        (ROOT / f"configs/seed-{seed}.yaml").write_text(text)\n    config_hashes = {str(p.relative_to(ROOT)):file_sha(p) for p in (ROOT/"configs").glob("seed-*.yaml")}\n    format_fn = formatter(tok)\n    all_rows = []\n    counts = {}\n    for split in ("train", "validation", "test"):\n        raw = rows(split)\n        counts[split] = len(raw)\n        for item in raw:\n            messages = item["messages"]\n            if any(token in message["content"] for token in tok.all_special_tokens for message in messages):\n                raise ValueError("reserved tokenizer token in source content")\n            functions = functions_from_messages(messages)\n            expected = parse_calls(messages[-1]["content"])\n            score = score_turn(messages[-1]["content"], expected, functions)\n            if not score["complete"]:\n                raise ValueError("invalid prepared reference")\n            for prefix, generation in ((messages, False), (messages[:-1], True)):\n                if tok.apply_chat_template(prefix, chat_template=native, tokenize=False, add_generation_prompt=generation) != tok.apply_chat_template(prefix, tokenize=False, add_generation_prompt=generation):\n                    raise ValueError("marked template changes native Qwen rendering")\n            encoded = audit_mask(messages, tok, format_fn)\n            all_rows.append({"id": digest(messages), "split": split, "messages": messages, "functions": functions,\n                             "expected": expected, "call_count": len(expected), "reference_tokens": len(encoded["input_ids"]),\n                             "supervised_tokens": sum(x != -100 for x in encoded["labels"][1:])})\n    if not 1000 <= counts["train"] <= 2000 or counts["validation"] < 100 or counts["test"] < 100:\n        raise ValueError("Prepared corpus does not meet experiment size requirements")\n    if len({row["id"] for row in all_rows}) != len(all_rows):\n        raise ValueError("duplicate example IDs")\n    group_metadata(all_rows)\n    write_jsonl(ROOT / "evidence/references.jsonl", all_rows)\n    write_jsonl(ROOT / "data/challenge.jsonl", challenge_rows())\n    validation = [row for row in all_rows if row["split"] == "validation"]\n    review_sample = sorted(validation,key=lambda row:digest([42,"reference-review",row["id"]]))[:12]\n    write_jsonl(ROOT/"evidence/validation-reference-review.jsonl",\n                [{"id":row["id"],"messages":row["messages"]} for row in review_sample])\n    longest_target = max(row["supervised_tokens"] for row in validation)\n    generation_cap = int(math.ceil((longest_target + 32) / 32) * 32)\n    if generation_cap > 1024:\n        raise ValueError("Validation requires >1024 new tokens; revise policy before training")\n    policy = {"created_utc": now(), "dataset": "pernavjain/tool-calling", "dataset_hashes": EXPECTED_DATA,\n        "model": MODEL_ID, "model_revision": MODEL_REV, "soup_commit": SOUP_REV,\n        "seeds": SETTINGS["seeds"], "allocator_config": SETTINGS["allocator_config"],\n        "selection": "within seed: lowest epoch validation loss; across seeds: validation complete-turn accuracy, then token-weighted NLL, then seed",\n        "trainer_policy": TRAINER_POLICY,\n        "generation_max_new_tokens": generation_cap, "generation_cap_source": "validation target lengths only +32, rounded to32",\n        "max_length": SETTINGS["max_length"], "bootstrap_replicates": 10000, "bootstrap_seed": 42,\n        "gain_min": .05, "quant_loss_max": .02, "quant_ci_lower_min": -.05, "general_regression_max": .05,\n        "headroom_bytes_min": 2**30, "merge_logprob_ceiling": .10, "merge_logprob_floor_budget": .02,\n        "merge_tolerance_formula": "max(0.02,10*measured_repeat_and_noop_floor), capped at0.10; frozen before test",\n        "merge_max_kl":.02,"weight_relative_residual_max":.25,"weight_absolute_residual_max":.001,\n        "base_disabled_logprob_floor":.002,"minimum_adapter_or_q_effect":1e-5,"effect_floor_multiplier":10,\n        "general_max_new_tokens":256,\n        "probe_rows": 4, "probe_positions": 4, "controls_required": True,\n        "config_hashes":config_hashes,\n        "source_labels": "not included in the four-file export; no invented per-source result slices",\n        "scope": "call-only single-turn function calling; structural reference audit, not semantic adjudication"}\n    write_json(ROOT / "run_policy.json", policy)\n    base_identity = model_manifest(base)\n    write_json(ROOT / "evidence/base-identity.json", base_identity)\n    initial_estimate = {"created_before_training": now(), "unique_parameters": 1543714304,\n        "lora_parameters_expected": 9232384, "base_fp16_gib": 1543714304 * 2 / 2**30,\n        "lora_fp32_parameter_gradient_adam_gib": 9232384 * 16 / 2**30,\n        "one_fp16_full_logits_gib": SETTINGS["max_length"] * 151936 * 2 / 2**30,\n        "one_fp32_full_logits_gib": SETTINGS["max_length"] * 151936 * 4 / 2**30,\n        "kv_cache_bytes_per_token_batch1": 28 * 2 * 2 * 128 * 2,\n        "nf4_storage_floor_gib": (1310195712 * .5 + 233373696 * 2) / 2**30,\n        "not_in_floor": ["checkpointed/recomputed activations", "attention/logit/loss workspaces", "allocator caches/context", "NF4 metadata", "host staging"],\n        "training_memory_preflight_budget_gib": 12.0,\n        "budget_label": "conservative planning envelope, not a prediction of actual peak",\n        "phase_model": "baseline/base+LoRA residency; forward/backward; lazy Adam state; eval/save; CPU merge; GPU NF4; cold load+KV inference"}\n    write_json(ROOT / "evidence/memory-estimate-before.json", initial_estimate)\n    training = base_config_mapping(fragment, SETTINGS["seeds"][0])["training"]\n    steps_per_epoch = math.ceil(counts["train"] / (training["batch_size"] * training["gradient_accumulation_steps"]))\n    write_json(ROOT / "evidence/training-plan.json", {\n        "training": training, "trainer_policy": TRAINER_POLICY, "seeds": SETTINGS["seeds"],\n        "examples_per_epoch": counts["train"], "optimizer_steps_per_epoch": steps_per_epoch,\n        "optimizer_steps_per_seed": steps_per_epoch * training["epochs"],\n        "rationale": "Two passes over the curated 1600-example set, lower 1e-4 learning rate and 5% warmup; retain the better validation-loss epoch. This recipe is not claimed optimal.",\n        "training_quantization": "none: FP16 base with LoRA; NF4 is a separate post-merge deployment experiment",\n    })\n    short = min(validation, key=lambda x: x["reference_tokens"])\n    fixtures = {"malformed_call": copy.deepcopy(short["messages"]), "missing_tools": copy.deepcopy(short["messages"]),\n                "double_template": copy.deepcopy(short["messages"]), "length_boundary": copy.deepcopy(max(validation, key=lambda x: x["reference_tokens"])["messages"])}\n    fixtures["malformed_call"][-1]["content"] = "<tool_call>{bad json}</tool_call>"\n    fixtures["missing_tools"][0]["content"] = "You are a helpful assistant."\n    fixtures["double_template"][1]["content"] = "<|im_start|>user\\n" + fixtures["double_template"][1]["content"] + "<|im_end|>"\n    for name, messages in fixtures.items():\n        write_jsonl(ROOT / f"controls/{name}.jsonl", [{"messages": messages}])\n    from trl.trainer.sft_trainer import DataCollatorForLanguageModeling\n    train = [row for row in all_rows if row["split"] == "train"]\n    encoded = [format_fn({"messages": row["messages"]}) for row in (min(train,key=lambda x:x["reference_tokens"]),max(train,key=lambda x:x["reference_tokens"]))]\n    assert len(encoded[0]["input_ids"]) < len(encoded[1]["input_ids"])\n    for pad in (tok.pad_token_id, tok.eos_token_id):\n        batch = DataCollatorForLanguageModeling(pad_token_id=pad, padding_free=False)(encoded)\n        for i, row in enumerate(encoded):\n            n = len(row["input_ids"])\n            assert batch["labels"][i,:n].tolist() == row["labels"]\n            assert (batch["labels"][i,n:] == -100).all().item()\n    summary = {"counts": counts, "max_length": SETTINGS["max_length"], "generation_cap": generation_cap,\n        "all_reference_masks_exact": True, "reserved_token_check": True, "split_overlap": 0, "collator_padding_checks": True,\n        "native_template_sha256": digest(native), "training_template_sha256": digest(tok.chat_template),\n        "group_counts": {s:len({x["group"] for x in all_rows if x["split"] == s}) for s in counts}}\n    write_json(ROOT / "evidence/data-ready.json", summary)\n    return {"outputs": ["evidence/data-ready.json", "evidence/base-identity.json", "run_policy.json",\n                        "evidence/references.jsonl", "data/challenge.jsonl", "evidence/memory-estimate-before.json",\n                        "evidence/training-plan.json"], **summary}\n\n\ndef clean_observation(value):\n    if isinstance(value, float) and not math.isfinite(value):\n        return {"nonfinite_observation": repr(value)}\n    if isinstance(value, dict):\n        return {key: clean_observation(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [clean_observation(item) for item in value]\n    return value\n\n\ndef reference_rows(split,tok=None):\n    if split != "challenge":\n        return [x for x in read_jsonl(ROOT / "evidence/references.jsonl") if x["split"] == split]\n    tok = tok or tokenizer()\n    fmt = formatter(tok)\n    result = []\n    for item in rows("challenge"):\n        messages = item["messages"]\n        encoded = audit_mask(messages, tok, fmt)\n        result.append({"id": digest(messages), "group": digest(messages), "split": split, "messages": messages,\n                       "functions": functions_from_messages(messages), "expected": parse_calls(messages[-1]["content"]),\n                       "call_count": len(parse_calls(messages[-1]["content"])), "reference_tokens": len(encoded["input_ids"])})\n    return result\n\n\ndef probe_rows():\n    candidates = reference_rows("validation")\n    ordered = sorted(candidates, key=lambda x: (x["reference_tokens"], x["id"]))\n    selected = [ordered[0], ordered[-1], max(candidates, key=lambda x:x["call_count"])]\n    selected += sorted(candidates, key=lambda x:digest([42,x["id"]]))\n    unique = {row["id"]: row for row in selected}\n    return list(unique.values())[:read_json(ROOT / "run_policy.json")["probe_rows"]]\n\n\ndef encoded_probe(model, tok, row):\n    torch = TORCH\n    encoded = audit_mask(row["messages"], tok, formatter(tok))\n    batch = {key: torch.tensor([value], device="cuda") for key, value in encoded.items()}\n    model.eval()\n    with torch.inference_mode():\n        output = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], use_cache=False)\n        positions = torch.nonzero(batch["labels"][0,1:] != -100).flatten()\n        targets = batch["labels"][0,positions + 1]\n        log_probs = output.logits[0,positions,:].float().log_softmax(-1)\n        reference = log_probs.gather(-1,targets[:,None]).flatten()\n        n = min(read_json(ROOT / "run_policy.json")["probe_positions"],len(positions))\n        choice = torch.linspace(0,len(positions)-1,n,device="cuda").long().unique()\n        return {"id": row["id"], "nll": float(-reference.mean()), "reference_log_probs": reference.cpu().tolist(),\n                "positions": positions[choice].cpu().tolist(), "distribution": log_probs[choice].cpu()}\n\n\ndef probe_model(model, tok):\n    return [encoded_probe(model, tok, row) for row in probe_rows()]\n\n\ndef compare_probes(before, after):\n    torch = TORCH\n    if not before or len(before) != len(after):\n        raise ValueError("probe counts differ")\n    errors, nll_errors, kls = [], [], []\n    for a,b in zip(before,after):\n        if a["id"] != b["id"] or a["positions"] != b["positions"]:\n            raise ValueError("probe identities differ")\n        if len(a["reference_log_probs"]) != len(b["reference_log_probs"]) or a["distribution"].shape != b["distribution"].shape:\n            raise ValueError("probe dimensions differ")\n        errors.extend(abs(x-y) for x,y in zip(a["reference_log_probs"],b["reference_log_probs"]))\n        nll_errors.append(abs(a["nll"]-b["nll"]))\n        kls.extend((a["distribution"].exp() * (a["distribution"]-b["distribution"])).sum(-1).tolist())\n    if not all(math.isfinite(v) for v in errors+nll_errors+kls):\n        raise ValueError("nonfinite probe comparison")\n    return {"max_reference_logprob_error": max(errors), "mean_reference_logprob_error": statistics.mean(errors),\n            "max_row_nll_difference": max(nll_errors), "mean_kl_before_after": statistics.mean(kls),\n            "max_kl_before_after": max(kls), "rows": len(before), "reference_tokens": len(errors), "distribution_positions":len(kls)}\n\n\ndef frozen_fingerprints(model):\n    result = {}\n    for name, parameter in model.named_parameters():\n        if not parameter.requires_grad and re.search(r"layers\\.(0|27)\\.self_attn\\.(q_proj|v_proj)\\.base_layer\\.weight$", name):\n            result[name] = hashlib.sha256(parameter.detach().cpu().contiguous().numpy().tobytes()).hexdigest()\n    if len(result) != 4:\n        raise ValueError("unexpected architecture: selected frozen projections not found")\n    return result\n\n\ndef train_stage(seed):\n    from soup_cli.data.loader import load_dataset\n    from soup_cli.trainer.sft import SFTTrainerWrapper\n    from soup_cli.bench.collector import BenchCollector\n    from transformers import TrainerCallback, set_seed\n    from dataclasses import asdict\n    benchmark_path = ROOT / "evidence/stages/bench-train.json"\n    benchmark = read_json(benchmark_path) if benchmark_path.is_file() else {}\n    if (benchmark.get("settings_sha256") != digest(SETTINGS)\n        or benchmark.get("runtime_sha256") != file_sha(__file__)\n        or not benchmark_headroom(benchmark, SETTINGS["allocator_config"])["passed"]):\n        raise RuntimeError("Primary training requires a passing T4 scratch memory gate under the same allocator policy.")\n    torch = cuda()\n    set_seed(seed)\n    if not check_bound_artifact(read_json(ROOT/"evidence/base-identity.json"),artifact_path("B")):\n        raise ValueError("Pristine base bytes changed before training")\n    config = cfg(seed)\n    out = ROOT / f"artifacts/adapter-{seed}"\n    if out.exists():\n        raise ValueError("Adapter output already exists; choose a new run after an interrupted training stage")\n    train_data = load_dataset(config.data)["train"]\n    val_data = load_dataset(config.data.model_copy(update={"train":"./data/validation.jsonl","val_split":0}))["train"]\n    for split,data in (("train",train_data),("validation",val_data)):\n        if [digest(x["messages"]) for x in data] != [digest(x["messages"]) for x in rows(split)]:\n            raise ValueError("Soup loader changed prepared split membership/order")\n    with CAPTURE.phase("base_load_and_lora_setup"):\n        wrapper = SFTTrainerWrapper(config, device="cuda", report_to="none")\n        wrapper.setup({"train":train_data,"val":val_data})\n        trainer = wrapper.trainer\n        configure_training(trainer)\n        model = trainer.model\n        assert trainer.args.fp16 and not trainer.args.bf16 and trainer.args.per_device_train_batch_size == 1\n        trainables = {name:p for name,p in model.named_parameters() if p.requires_grad}\n        assert sum(p.numel() for p in trainables.values()) == 9232384\n        assert sum(p.numel() for p in model.parameters()) - 9232384 == 1543714304\n        assert all("lora_" in name for name in trainables)\n        assert model.get_input_embeddings().weight.data_ptr() == model.get_output_embeddings().weight.data_ptr()\n        assert all(not p.is_meta and p.device.type == "cuda" for p in model.parameters())\n        before = {name:p.detach().cpu().clone() for name,p in trainables.items()}\n        frozen_before = frozen_fingerprints(model)\n        initial_dir = ROOT / f"artifacts/initial-{seed}"\n        model.save_pretrained(initial_dir, safe_serialization=True)\n        wrapper.tokenizer.save_pretrained(initial_dir)\n    with CAPTURE.phase("initial_noop_and_numerical_floor"):\n        first = seed == SETTINGS["seeds"][0]\n        with model.disable_adapter():\n            base_probe = probe_model(model, wrapper.tokenizer)\n            repeated = [probe_model(model, wrapper.tokenizer) for _ in range(2)] if first else []\n        zero_probe = probe_model(model, wrapper.tokenizer)\n        zero_error = compare_probes(base_probe,zero_probe)["max_reference_logprob_error"]\n        repeat_error = max((compare_probes(base_probe,x)["max_reference_logprob_error"] for x in repeated),default=0.)\n        if zero_error > .02:\n            raise ValueError("Initialized adapter is not a no-op within the predeclared floor ceiling")\n        floor = max(zero_error,repeat_error)\n        if first:\n            calibration = {"utc_before_training":now(),"repeat_base_runs":3,"repeat_error":repeat_error,\n                "initialized_adapter_error":zero_error,"floor":floor,"merge_reference_logprob_tolerance":max(.02,10*floor),\n                "ceiling":.10,"scope":"four fixed validation rows; deterministic baseline under actual kernels"}\n            if calibration["merge_reference_logprob_tolerance"] > calibration["ceiling"]:\n                raise ValueError("Numerical instrument exceeds frozen tolerance ceiling")\n            write_json(ROOT / "evidence/numerical-calibration.json",calibration)\n    collector = BenchCollector(warmup_steps=3)\n    collector._sync = torch.cuda.synchronize\n    trainer.add_callback(collector)\n    observed = {"microbatches":0,"shifted_supervised_tokens":0,"nonpadding_tokens":0,"padded_tokens":0,\n                "optimizer_callbacks":0,"successful_nonzero_lr_steps":0,"amp_skipped_steps":0,"first_materialized_update":None}\n    probe_parameter = next(p for name,p in trainables.items() if "lora_B" in name)\n    first_forward_hook = None\n    original_step = trainer.training_step\n\n    def training_step(current_model, inputs, *args, **kwargs):\n        collector.observe_batch(inputs)\n        observed["microbatches"] += 1\n        observed["shifted_supervised_tokens"] += int((inputs["labels"][:,1:] != -100).sum().item())\n        observed["nonpadding_tokens"] += int(inputs["attention_mask"].sum().item())\n        observed["padded_tokens"] += inputs["input_ids"].numel()\n        value = original_step(current_model,inputs,*args,**kwargs)\n        if observed["microbatches"] == 1:\n            CAPTURE.mark("first_backward_complete")\n        return value\n    trainer.training_step = training_step\n\n    def first_forward(_model,_inputs,_output):\n        CAPTURE.mark("first_forward_complete")\n        first_forward_hook.remove()\n    first_forward_hook = model.register_forward_hook(first_forward)\n\n    class UpdateEvidence(TrainerCallback):\n        def __init__(self):\n            self.sample = None\n            self.lr = None\n            self.first_optimizer = True\n\n        def on_train_begin(self,args,state,control,**kwargs):\n            optimizer = kwargs["optimizer"]\n            wanted = {id(p) for p in model.parameters() if p.requires_grad}\n            actual = {id(p) for group in optimizer.param_groups for p in group["params"]}\n            assert wanted == actual, "Optimizer membership does not match trainable parameters"\n            observed["optimizer_membership_exact"] = True\n\n        def on_pre_optimizer_step(self,args,state,control,**kwargs):\n            self.lr = max(group["lr"] for group in kwargs["optimizer"].param_groups)\n            if observed["first_materialized_update"] is None and self.lr > 0:\n                self.sample = probe_parameter.detach().clone()\n            if self.first_optimizer:\n                CAPTURE.mark("before_first_optimizer_step")\n\n        def on_optimizer_step(self,args,state,control,**kwargs):\n            observed["optimizer_callbacks"] += 1\n            skipped = bool(trainer.accelerator.optimizer_step_was_skipped)\n            observed["amp_skipped_steps"] += int(skipped)\n            if not skipped and self.lr > 0:\n                observed["successful_nonzero_lr_steps"] += 1\n                if self.sample is not None:\n                    delta = float((probe_parameter.detach()-self.sample).abs().max())\n                    if delta > 0 and observed["first_materialized_update"] is None:\n                        observed["first_materialized_update"] = {"global_step_before_increment":state.global_step,"lr":self.lr,"max_abs_delta":delta}\n            self.sample = None\n            if self.first_optimizer:\n                state_bytes = collections.Counter()\n                for state_values in kwargs["optimizer"].state.values():\n                    for value in state_values.values():\n                        if isinstance(value,torch.Tensor):\n                            state_bytes[str(value.dtype)+"@"+str(value.device)] += value.numel()*value.element_size()\n                observed["optimizer_state_bytes_after_first_attempt"] = dict(state_bytes)\n                CAPTURE.mark("after_first_optimizer_step")\n                self.first_optimizer = False\n\n        def on_log(self,args,state,control,logs=None,**kwargs):\n            event("training_log",seed=seed,step=state.global_step,logs=clean_observation(logs or {}))\n            if state.global_step and state.global_step % 20 == 0:\n                event("progress",stage=f"train-{seed}",optimizer_step=state.global_step)\n\n        def on_evaluate(self,args,state,control,**kwargs):\n            CAPTURE.mark(f"evaluation_complete_step_{state.global_step}")\n\n        def on_save(self,args,state,control,**kwargs):\n            CAPTURE.mark(f"checkpoint_saved_step_{state.global_step}")\n\n    trainer.add_callback(UpdateEvidence())\n    original_save = trainer.save_model\n    def observed_save(*args,**kwargs):\n        with CAPTURE.phase("model_save"):\n            return original_save(*args,**kwargs)\n    trainer.save_model = observed_save\n    with CAPTURE.phase("primary_training"):\n        summary = wrapper.train(run_id=f"seed-{seed}")\n    # Keep Trainer optimizer/checkpoint files outside the clean deployable adapter.\n    with CAPTURE.phase("clean_adapter_export"):\n        model.save_pretrained(out,safe_serialization=True)\n        wrapper.tokenizer.save_pretrained(out)\n    checkpoint = checkpoint_evidence(trainer, out)\n    assert collector.steps and observed["first_materialized_update"] is not None\n    assert observed["successful_nonzero_lr_steps"] > 0 and observed["optimizer_membership_exact"]\n    assert all(torch.isfinite(p).all().item() for p in trainables.values())\n    deltas = {name:float((p.detach().cpu().float()-before[name].float()).abs().max()) for name,p in trainables.items()}\n    assert max(deltas.values()) > 0\n    frozen_after = frozen_fingerprints(model)\n    assert frozen_after == frozen_before\n    with CAPTURE.phase("trained_adapter_on_off"):\n        active = probe_model(model,wrapper.tokenizer)\n        with model.disable_adapter():\n            disabled = probe_model(model,wrapper.tokenizer)\n        effect = compare_probes(disabled,active)\n        disabled_error = compare_probes(base_probe,disabled)\n    initial_manifest = model_manifest(initial_dir)\n    final_manifest = model_manifest(out)\n    assert initial_manifest["sha256"] != final_manifest["sha256"]\n    counted = collector.steps[3:]\n    result = {"seed":seed,"training_summary":summary,"checkpoint_selection":checkpoint,\n        "trainer_log_history":clean_observation(trainer.state.log_history),\n        "trainable_shapes_dtypes":{name:{"shape":list(p.shape),"dtype":str(p.dtype),"device":str(p.device)} for name,p in trainables.items()},\n        "base_unique_parameters":1543714304,"lora_parameters":9232384,"embedding_and_head_tied":True,\n        "resolved_training_args":trainer.args.to_dict(),"observations":observed,"changed_adapter_tensors":sum(x>0 for x in deltas.values()),\n        "max_parameter_delta":max(deltas.values()),"frozen_projection_hashes_before":frozen_before,"frozen_projection_hashes_after":frozen_after,\n        "adapter_effect":effect,"disabled_vs_pristine":disabled_error,"initial":initial_manifest,"final":final_manifest,\n        "bench_collector":{"steps":[asdict(x) for x in collector.steps],"param_snapshot":asdict(collector.param_snapshot()),\n            "postwarmup_seconds":sum(x.wall_seconds for x in counted),"postwarmup_useful_tokens_unshifted":sum(x.useful_tokens for x in counted),\n            "timing_scope":"instrumented synchronized optimizer intervals, including between-step checkpoint/data overhead; full wall time separate"}}\n    result = clean_observation(result)\n    destination = ROOT / f"evidence/training-{seed}.json"\n    write_json(destination,result)\n    write_json(out / "provenance.json",{"seed":seed,"model_revision":MODEL_REV,"base_identity":read_json(ROOT/"evidence/base-identity.json")["sha256"],\n        "training_data_sha256":EXPECTED_DATA["train.jsonl"],"soup_commit":SOUP_REV,"runtime_sha256":file_sha(__file__),"adapter_sha256":final_manifest["sha256"]})\n    return {"outputs":[str(destination.relative_to(ROOT)),str((out/"provenance.json").relative_to(ROOT)),\n                       str((out/"adapter_model.safetensors").relative_to(ROOT)),str((initial_dir/"adapter_model.safetensors").relative_to(ROOT))],\n            "global_steps":trainer.state.global_step,"selected_epoch":checkpoint["selected_epoch"],"updated":True,"seed":seed}\n\n\ndef artifact_path(variant,seed=None):\n    if variant == "B":\n        return ROOT / "artifacts/base"\n    if variant in {"A","A0"}:\n        return ROOT / f"artifacts/{\'adapter\' if variant==\'A\' else \'initial\'}-{seed}"\n    return ROOT / {"M":"artifacts/merged-fp16","Q":"artifacts/merged-nf4","Q0":"artifacts/control-q0"}[variant]\n\n\ndef load_variant(variant, seed=None, portable=False):\n    from transformers import AutoModelForCausalLM\n    path = artifact_path(variant,seed)\n    if variant == "B":\n        bound = read_json(ROOT/"evidence/base-identity.json")\n    elif variant in {"A","A0"}:\n        training = read_json(ROOT/f"evidence/training-{seed}.json")\n        bound = training["final" if variant == "A" else "initial"]\n        if not check_bound_artifact(read_json(ROOT/"evidence/base-identity.json"),artifact_path("B")):\n            raise ValueError("Adapter base bytes changed")\n    else:\n        bound = read_json(ROOT/f"evidence/artifact-{variant}.json")["artifact"]\n    if not check_bound_artifact(bound,path):\n        raise ValueError("Artifact bytes drifted from their recorded identity")\n    if variant in {"A","A0"}:\n        from peft import PeftModel\n        base = AutoModelForCausalLM.from_pretrained(artifact_path("B"),dtype=TORCH.float16,device_map={"":"cuda:0"},local_files_only=True)\n        model = PeftModel.from_pretrained(base,path,is_trainable=False,local_files_only=True)\n    else:\n        if portable:\n            isolated = ROOT / "artifacts/portable" / variant\n            isolated.mkdir(parents=True,exist_ok=True)\n            for file in path.iterdir():\n                if file.is_file() and file.name != "provenance.json":\n                    target = isolated / file.name\n                    if not target.exists():\n                        os.link(file,target)\n            forbidden = [str((ROOT/"artifacts/base").resolve())] + [str(p.resolve()) for p in (ROOT/"artifacts").glob("adapter-*")]\n            def deny_base_reads(event_name,args):\n                if event_name == "open" and args and isinstance(args[0],(str,bytes,os.PathLike)):\n                    candidate = os.fsdecode(args[0])\n                    if any(candidate == prefix or candidate.startswith(prefix+os.sep) for prefix in forbidden):\n                        raise RuntimeError("Portable load attempted to read a base/adapter path")\n            sys.addaudithook(deny_base_reads)\n            path = isolated\n        model = AutoModelForCausalLM.from_pretrained(path,dtype=TORCH.float16,device_map={"":"cuda:0"},local_files_only=True)\n    model.eval()\n    if not all(not p.is_meta and p.device.type == "cuda" for p in model.parameters()):\n        raise ValueError("CPU/disk/meta fallback detected")\n    from bitsandbytes.nn import Linear4bit\n    quantized = sum(isinstance(module,Linear4bit) for module in model.modules())\n    if variant in {"Q","Q0"}:\n        quant = model.config.quantization_config\n        quant = quant.to_dict() if hasattr(quant,"to_dict") else quant\n        assert quantized == 196 and quant["load_in_4bit"] and quant["bnb_4bit_quant_type"] == "nf4" and quant["bnb_4bit_use_double_quant"]\n        assert all(module.weight.quant_state is not None for module in model.modules() if isinstance(module,Linear4bit))\n        assert all(module.compute_dtype == TORCH.float16 and getattr(module.weight.quant_state,"state2",None) is not None\n                   for module in model.modules() if isinstance(module,Linear4bit))\n    else:\n        quant = None\n        assert quantized == 0\n    info = {"variant":variant,"resolved_path":str(path),"class":type(model).__name__,"quantized_linear_count":quantized,\n            "quantization_config":quant,"devices":sorted({str(p.device) for p in model.parameters()}),\n            "parameter_dtypes":dict(collections.Counter(str(p.dtype) for p in model.parameters())),"portable_base_adapter_reads_blocked":portable}\n    return model, info\n\n\ndef generate(model,tok,messages,max_new_tokens,general=False):\n    template = (ROOT/"evidence/native-template.jinja").read_text() if general else tok.chat_template\n    encoded = tok.apply_chat_template(messages,chat_template=template,tokenize=True,add_generation_prompt=True,return_tensors="pt",return_dict=True).to("cuda")\n    TORCH.cuda.synchronize()\n    start = time.monotonic()\n    with TORCH.inference_mode():\n        output = model.generate(**encoded,max_new_tokens=max_new_tokens,do_sample=False,use_cache=True,\n                                eos_token_id=tok.eos_token_id,pad_token_id=tok.pad_token_id)\n    TORCH.cuda.synchronize()\n    generated = output[0,encoded["input_ids"].shape[-1]:].cpu().tolist()\n    ended = bool(generated and generated[-1] == tok.eos_token_id)\n    text_ids = generated[:-1] if ended else generated\n    return {"text":tok.decode(text_ids,skip_special_tokens=False,clean_up_tokenization_spaces=False),\n            "raw_text":tok.decode(generated,skip_special_tokens=False,clean_up_tokenization_spaces=False),\n            "generated_ids":generated,"generated_tokens":len(generated),"prompt_tokens":encoded["input_ids"].shape[-1],\n            "truncated":not ended,"generation_seconds":time.monotonic()-start,"stop":"eot" if ended else "length_or_other_stop"}\n\n\ndef teacher_nll(model,tok,messages):\n    batch = audit_mask(messages,tok,formatter(tok))\n    tensors = {key:TORCH.tensor([value],device="cuda") for key,value in batch.items()}\n    with TORCH.inference_mode():\n        output = model(input_ids=tensors["input_ids"],attention_mask=tensors["attention_mask"],use_cache=False)\n        positions = TORCH.nonzero(tensors["labels"][0,1:] != -100).flatten()\n        targets = tensors["labels"][0,positions+1]\n        probabilities = output.logits[0,positions,:].float().log_softmax(-1)\n        loss_sum = float(-probabilities.gather(-1,targets[:,None]).sum())\n    if not math.isfinite(loss_sum):\n        raise ValueError("nonfinite assistant NLL")\n    return loss_sum,len(positions)\n\n\ndef evaluate_stage(variant,split,seed=None,general=False,probes_only=False):\n    from soup_cli.eval.custom import tool_call_match\n    cuda()\n    if split == "test" and not (ROOT/"evidence/policy-frozen-before-test.json").exists():\n        raise ValueError("Freeze selection, calibration and policy before test evaluation")\n    if split == "test":\n        frozen = read_json(ROOT/"evidence/policy-frozen-before-test.json")\n        if file_sha(ROOT/"run_policy.json") != frozen["policy_sha256"] or file_sha(ROOT/"data/test.jsonl") != frozen["test_hash"]:\n            raise ValueError("Frozen test/policy bytes changed")\n        if file_sha(ROOT/"evidence/references.jsonl") != frozen["references_hash"]:\n            raise ValueError("Frozen evaluation reference bytes changed")\n    tok = tokenizer(artifact_path(variant)) if variant in {"M","Q","Q0"} else tokenizer()\n    items = reference_rows(split,tok) if not probes_only else []\n    with CAPTURE.phase("cold_load"):\n        model, identity = load_variant(variant,seed,portable=variant in {"M","Q"})\n    label = variant + (f"-{seed}" if seed is not None else "")\n    outputs = []\n    if split == "validation":\n        with CAPTURE.phase("fixed_validation_distribution_probes"):\n            probes = probe_model(model,tok)\n            probe_path = ROOT / f"evidence/probes/{split}-{label}.pt"\n            probe_path.parent.mkdir(parents=True,exist_ok=True)\n            TORCH.save(probes,probe_path)\n        outputs.append(str(probe_path.relative_to(ROOT)))\n    with CAPTURE.phase("generation_warmup_excluded_from_latency"):\n        generate(model,tok,probe_rows()[0]["messages"][:-1],8)\n    if not probes_only:\n        predictions = []\n        policy = read_json(ROOT/"run_policy.json")\n        path = ROOT / f"evidence/predictions/{split}-{label}.jsonl"\n        path.parent.mkdir(parents=True,exist_ok=True)\n        with CAPTURE.phase("teacher_forcing_and_greedy_generation"),path.open("w") as handle:\n            for i,row in enumerate(items):\n                loss_sum,count = teacher_nll(model,tok,row["messages"])\n                generated = generate(model,tok,row["messages"][:-1],policy["generation_max_new_tokens"])\n                scored = score_turn(generated["text"],row["expected"],row["functions"],generated["truncated"],tool_call_match)\n                record = {"id":row["id"],"group":row["group"],"variant":variant,"seed":seed,\n                          "call_count":row["call_count"],"reference_tokens":row["reference_tokens"],\n                          "nll_sum":loss_sum,"nll_tokens":count,**generated,**scored}\n                handle.write(canonical(record)+"\\n");handle.flush()\n                predictions.append(record)\n                if (i+1)%10 == 0:\n                    event("progress",stage=f"eval-{split}-{label}",completed=i+1,total=len(items))\n        summary = {"identity":identity,"split":split,"variant":variant,"seed":seed,**summarize_predictions(predictions)}\n        destination = ROOT / f"evidence/scores/{split}-{label}.json"\n        write_json(destination,summary)\n        outputs += [str(path.relative_to(ROOT)),str(destination.relative_to(ROOT))]\n    if general:\n        from soup_cli.eval.gate_suites import DEFAULT_GENERAL_SUITE, BUNDLED_SCORER_REVISION, BUNDLED_SCORER_FINGERPRINT, score_bundled_suite\n        suites = {}\n        raw = []\n        generation_errors = []\n        with CAPTURE.phase("bundled_general_suites"):\n            for suite in DEFAULT_GENERAL_SUITE:\n                def gen(prompt):\n                    try:\n                        result = generate(model,tok,[{"role":"system","content":"You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},\n                                                     {"role":"user","content":prompt}],256,general=True)\n                    except Exception as exc:\n                        generation_errors.append({"suite":suite,"prompt":prompt,"error":f"{type(exc).__name__}: {exc}"})\n                        return None\n                    raw.append({"suite":suite,"prompt":prompt,**result})\n                    return None if result["truncated"] else result["text"]\n                suites[suite] = score_bundled_suite(suite,gen)\n                event("progress",stage=f"general-{label}",suite=suite,score=suites[suite])\n        destination = ROOT / f"evidence/general/{label}.json"\n        write_jsonl(destination.with_suffix(".jsonl"),[*raw,*generation_errors])\n        if generation_errors:\n            raise RuntimeError(f"{len(generation_errors)} bundled-suite generation errors; see {destination.with_suffix(\'.jsonl\')}")\n        write_json(destination,{"scores":suites,"scorer_revision":BUNDLED_SCORER_REVISION,"scorer_fingerprint":BUNDLED_SCORER_FINGERPRINT,\n                                "counts":dict(collections.Counter(x["suite"] for x in raw)),"generation_budget":256,"native_template":True})\n        outputs += [str(destination.relative_to(ROOT)),str(destination.with_suffix(".jsonl").relative_to(ROOT))]\n    write_json(ROOT/f"evidence/identity-{label}.json",identity)\n    return {"outputs":[*outputs,f"evidence/identity-{label}.json"],"identity":identity}\n\n\ndef select_stage():\n    candidates = []\n    for seed in SETTINGS["seeds"]:\n        score = read_json(ROOT/f"evidence/scores/validation-A-{seed}.json")\n        training = read_json(ROOT/f"evidence/training-{seed}.json")\n        assert training["observations"]["first_materialized_update"] is not None\n        candidates.append({"seed":seed,"complete":score["complete"],"nll":score["mean_assistant_nll"],\n                           "selected_epoch":training["checkpoint_selection"]["selected_epoch"],\n                           "best_validation_loss":training["checkpoint_selection"]["best_validation_loss"],\n                           "adapter_sha256":training["final"]["sha256"]})\n    selected = sorted(candidates,key=lambda x:(-x["complete"],x["nll"],x["seed"]))[0]\n    result = {"selected":selected,"all_seeds":candidates,"selection_uses_test":False,"utc":now()}\n    write_json(ROOT/"evidence/selection.json",result)\n    return {"outputs":["evidence/selection.json"],**result}\n\n\ndef merge_stage(variant):\n    cuda()\n    selected = read_json(ROOT/"evidence/selection.json")["selected"]["seed"]\n    seed = SETTINGS["seeds"][0] if variant == "Q0" else selected\n    adapter = artifact_path("A0" if variant == "Q0" else "A",seed)\n    if not check_bound_artifact(read_json(ROOT/"evidence/base-identity.json"),artifact_path("B")):\n        raise ValueError("Wrong/stale base before merge")\n    expected_adapter = read_json(ROOT/f"evidence/training-{seed}.json")["initial" if variant == "Q0" else "final"]\n    if not check_bound_artifact(expected_adapter,adapter):\n        raise ValueError("Wrong/stale adapter before merge")\n    output = artifact_path(variant)\n    if output.exists():\n        raise ValueError("Export destination exists after a partial stage; use a fresh run or inspect/archive that destination explicitly")\n    args = ["merge","--adapter",str(adapter),"--base",str(artifact_path("B")),"--output",str(output),"--dtype","float16"]\n    if variant in {"Q","Q0"}:\n        args += ["--save-format","4bit"]\n    with CAPTURE.phase("cpu_dense_merge" if variant == "M" else "dense_staging_nf4_save"):\n        run_soup(args)\n    if not list(output.glob("*.safetensors")) or (output/"adapter_config.json").exists():\n        raise ValueError("Export is not a standalone saved model")\n    run_soup(["adapters","check-safetensors",str(output),"--strict"])\n    identity = model_manifest(output)\n    provenance = {"variant":variant,"utc":now(),"base":read_json(ROOT/"evidence/base-identity.json"),\n                  "adapter":model_manifest(adapter),"artifact":identity,"runtime_sha256":file_sha(__file__),"soup_commit":SOUP_REV}\n    write_json(output/"provenance.json",provenance)\n    write_json(ROOT/f"evidence/artifact-{variant}.json",provenance)\n    return {"outputs":[f"evidence/artifact-{variant}.json",str((output/"provenance.json").relative_to(ROOT)),\n                       *[str(p.relative_to(ROOT)) for p in output.glob("*.safetensors")]],"variant":variant,"artifact_sha256":identity["sha256"]}\n\n\ndef tensor_from(directory,key=None,suffix=None):\n    from safetensors import safe_open\n    for filename in sorted(Path(directory).glob("*.safetensors")):\n        with safe_open(filename,framework="pt",device="cpu") as handle:\n            for name in handle.keys():\n                if (key is not None and name == key) or (suffix is not None and name.endswith(suffix)):\n                    return handle.get_tensor(name)\n    raise KeyError(key or suffix)\n\n\ndef weight_comparison(candidate_dir,adapter_dir):\n    config = read_json(adapter_dir/"adapter_config.json")\n    scale = config["lora_alpha"] / config["r"]\n    comparisons = []\n    for layer in (0,27):\n        for projection in ("q_proj","v_proj"):\n            prefix = f"model.layers.{layer}.self_attn.{projection}"\n            # The Hub checkpoint can be BF16; the measured merge starts from FP16.\n            base = tensor_from(artifact_path("B"),key=prefix+".weight").half()\n            a = tensor_from(adapter_dir,suffix=prefix+".lora_A.weight")\n            b = tensor_from(adapter_dir,suffix=prefix+".lora_B.weight")\n            expected = (base.float()+(b.float()@a.float())*scale).to(base.dtype).float()\n            candidate = tensor_from(candidate_dir,key=prefix+".weight").float()\n            effect = float((expected-base.float()).norm())\n            residual = float((candidate-expected).norm())\n            comparisons.append({"projection":prefix,"expected_effect_norm":effect,"residual_norm":residual,\n                                "relative_residual":residual/effect if effect else None,\n                                "max_abs_residual":float((candidate-expected).abs().max())})\n    policy = read_json(ROOT/"run_policy.json")\n    passed = all(x["relative_residual"] is not None\n        and x["relative_residual"] <= policy["weight_relative_residual_max"]\n        and x["max_abs_residual"] <= policy["weight_absolute_residual_max"] for x in comparisons)\n    return {"passed":passed,"selected_projections":comparisons,"scope":"four projections; float32 BA then FP16 rounding; 25% relative and0.001 absolute ceilings"}\n\n\ndef load_probes(name):\n    return TORCH.load(ROOT/f"evidence/probes/{name}.pt",map_location="cpu",weights_only=True)\n\n\ndef fidelity_stage():\n    cuda()\n    seed = read_json(ROOT/"evidence/selection.json")["selected"]["seed"]\n    calibration = read_json(ROOT/"evidence/numerical-calibration.json")\n    policy = read_json(ROOT/"run_policy.json")\n    a = load_probes(f"validation-A-{seed}")\n    b = load_probes("validation-B")\n    m = load_probes("validation-M")\n    q = load_probes("validation-Q")\n    q0 = load_probes("validation-Q0")\n    am = compare_probes(a,m)\n    mq = compare_probes(m,q)\n    bq0 = compare_probes(b,q0)\n    q0q = compare_probes(q0,q)\n    weights = weight_comparison(artifact_path("M"),artifact_path("A",seed))\n    wrong_base_export = weight_comparison(artifact_path("B"),artifact_path("A",seed))\n    q_identity = read_json(ROOT/"evidence/artifact-Q.json")["artifact"]\n    wrong_q0_rejected = not check_bound_artifact(q_identity,artifact_path("Q0"))\n    result = {"calibration":calibration,"A_to_M":am,"M_to_Q":mq,"B_to_Q0":bq0,"Q0_to_Q":q0q,"weights":weights,\n        "merge_faithful":weights["passed"] and am["max_reference_logprob_error"] <= calibration["merge_reference_logprob_tolerance"]\n                         and am["max_row_nll_difference"] <= calibration["merge_reference_logprob_tolerance"]\n                         and am["max_kl_before_after"] <= policy["merge_max_kl"],\n        "controls":{"base_substituted_for_merged_rejected":not wrong_base_export["passed"],"wrong_base_weight_details":wrong_base_export,\n                    "q0_substituted_for_q_identity_rejected":wrong_q0_rejected,\n                    "q_distinguishable_from_q0":q0q["max_reference_logprob_error"] > max(\n                        policy["minimum_adapter_or_q_effect"],policy["effect_floor_multiplier"]*calibration["floor"]),\n                    "q0_is_not_a_quantization_correction":"B/Q0 and M/Q are reported independently; errors are nonlinear"}}\n    write_json(ROOT/"evidence/fidelity-and-export-controls.json",result)\n    return {"outputs":["evidence/fidelity-and-export-controls.json"],"merge_faithful":result["merge_faithful"]}\n\n\ndef selftest_stage():\n    from soup_cli.eval.custom import tool_call_match\n    functions = {"f":{"name":"f","parameters":{"type":"object","properties":{"x":{"type":"integer"},"enabled":{"type":"boolean"}},\n                                                   "required":["x","enabled"],"additionalProperties":False}}}\n    reference = [{"name":"f","arguments":{"x":1,"enabled":True}}]\n    correct = tagged(reference)\n    variants = {\n        "correct":(correct,True), "numeric_json_semantics":(tagged([{"name":"f","arguments":{"x":1.0,"enabled":True}}]),True),\n        "wrong_argument":(tagged([{"name":"f","arguments":{"x":2,"enabled":True}}]),False),\n        "boolean_to_number":(tagged([{"name":"f","arguments":{"x":1,"enabled":1}}]),False),\n        "number_to_string":(tagged([{"name":"f","arguments":{"x":"1","enabled":True}}]),False),\n        "extra_call":(correct+"\\n"+correct,False), "prose":("Here: "+correct,False),\n        "unknown_tool":(tagged([{"name":"missing","arguments":{"x":1,"enabled":True}}]),False),\n        "duplicate_key":(\'<tool_call>{"name":"f","name":"f","arguments":{}}</tool_call>\',False),\n        "nonfinite":(\'<tool_call>{"name":"f","arguments":{"x":NaN}}</tool_call>\',False),\n        "escaped_newline":(\'<tool_call>\\\\n{"name":"f","arguments":{}}\\\\n</tool_call>\',False),\n        "incomplete":("<tool_call>{}",False),\n    }\n    results = {}\n    for name,(text,wanted) in variants.items():\n        scored = score_turn(text,reference,functions,matcher=tool_call_match)\n        assert scored["complete"] is wanted,(name,scored)\n        assert scored["complete"] == score_turn(text,reference,functions)["complete"],"Soup matcher and outer policy disagree"\n        results[name] = scored\n    assert not score_turn(correct,reference,functions,truncated=True,matcher=tool_call_match)["complete"]\n    multi = reference + [{"name":"f","arguments":{"x":2,"enabled":False}}]\n    assert score_turn(tagged(multi),multi,functions,matcher=tool_call_match)["complete"]\n    assert not score_turn(tagged(list(reversed(multi))),multi,functions,matcher=tool_call_match)["complete"]\n    tok = tokenizer()\n    fmt = formatter(tok)\n    independent = {}\n    for name in ("malformed_call","missing_tools","double_template","length_boundary"):\n        messages = read_jsonl(ROOT/f"controls/{name}.jsonl")[0]["messages"]\n        rejected = False\n        try:\n            functions_from_messages(messages)\n            parse_calls(messages[-1]["content"])\n            if any(token in m["content"] for token in tok.all_special_tokens for m in messages):\n                raise ValueError("reserved token")\n            if name == "length_boundary":\n                if len(tok.apply_chat_template(messages,tokenize=True,return_dict=False)) > 16:\n                    raise ValueError("would truncate at16")\n            audit_mask(messages,tok,fmt)\n        except ValueError:\n            rejected = True\n        assert rejected,name\n        independent[name] = "observed_caught"\n    write_json(ROOT/"evidence/scorer-and-data-controls.json",{"all_passed":True,"scorer_fixtures":results,"independent_data_controls":independent,\n        "multiple_call_order_control":True,"stop_failure_control":True,"Soup_scorer_reused":True})\n    return {"outputs":["evidence/scorer-and-data-controls.json"],"fixtures_passed":len(results)+2}\n\n\ndef freeze_stage():\n    policy = read_json(ROOT/"run_policy.json")\n    selection = read_json(ROOT/"evidence/selection.json")\n    required = ["evidence/numerical-calibration.json","evidence/fidelity-and-export-controls.json","evidence/scorer-and-data-controls.json"]\n    if any(not (ROOT/p).exists() for p in required):\n        raise ValueError("Development checks missing before test freeze")\n    frozen = {"frozen_utc":now(),"policy_sha256":file_sha(ROOT/"run_policy.json"),"policy":policy,"selection":selection,\n              "calibration":read_json(ROOT/required[0]),"artifacts":{v:read_json(ROOT/f"evidence/artifact-{v}.json")["artifact"] for v in ("M","Q","Q0")},\n              "test_hash":file_sha(ROOT/"data/test.jsonl"),"challenge_hash":file_sha(ROOT/"data/challenge.jsonl"),\n              "references_hash":file_sha(ROOT/"evidence/references.jsonl"),"rule_changes_after_test_allowed":False}\n    write_json(ROOT/"evidence/policy-frozen-before-test.json",frozen)\n    return {"outputs":["evidence/policy-frozen-before-test.json"]}\n\n\ndef soup_coverage():\n    names = {"malformed_call":"fixture-malformed-validate","missing_tools":"fixture-missing-doctor",\n             "double_template":"fixture-double-validate","length_boundary":"fixture-length-doctor","no_op_adapter":"initial-adapter-check"}\n    result = []\n    for failure,label in names.items():\n        path = ROOT/f"evidence/stages/{label}.json"\n        observed = read_json(path) if path.exists() else {}\n        code = observed.get("result",{}).get("exit_code")\n        status = "NOT_RUN" if code is None else "observed_missed" if code == 0 else "observed_caught" if code == 2 else "UNRESOLVED_RUNTIME_OR_USAGE_FAILURE"\n        result.append({"failure":failure,"Soup_status":status,\n                       "Soup_exit_code":code,"stage_report":str(path.relative_to(ROOT)),\n                       "scope_note":"Format validation does not validate embedded function-call semantics; narrow checks are not automatically Soup bugs."})\n    return result\n\n\ndef finalize_stage():\n    required = ["evidence/policy-frozen-before-test.json","evidence/fidelity-and-export-controls.json","evidence/scorer-and-data-controls.json",\n                *[f"evidence/predictions/test-{v}.jsonl" for v in ("B","M","Q")],\n                *[f"evidence/general/{v}.json" for v in ("B","M","Q")],\n                *[f"evidence/predictions/challenge-{v}.jsonl" for v in ("B","M","Q")]]\n    missing = [p for p in required if not (ROOT/p).exists()]\n    coverage = soup_coverage()\n    write_json(ROOT/"evidence/silent-failure-coverage.json",coverage)\n    if missing:\n        verdict = {"verdict":"DON\'T SHIP","gates":{"required_evidence_complete":False},"blockers":["Missing: "+p for p in missing],"metrics":None}\n        write_json(ROOT/"verdict.json",verdict)\n        return {"outputs":["verdict.json","evidence/silent-failure-coverage.json"],"verdict":"DON\'T SHIP","missing":missing}\n    frozen = read_json(ROOT/required[0])\n    fidelity = read_json(ROOT/required[1])\n    selected = frozen["selection"]["selected"]["seed"]\n    training = read_json(ROOT/f"evidence/training-{selected}.json")\n    predictions = {v:read_jsonl(ROOT/f"evidence/predictions/test-{v}.jsonl") for v in ("B","M","Q")}\n    refs = reference_rows("test")\n    expected_ids = [row["id"] for row in refs]\n    if len(expected_ids) != len(rows("test")) or any([row["id"] for row in result] != expected_ids for result in predictions.values()):\n        raise ValueError("Predictions must cover every frozen test turn in order")\n    pairs = {"Q_minus_B":grouped_pairs(predictions["B"],predictions["Q"]),"Q_minus_M":grouped_pairs(predictions["M"],predictions["Q"]),\n             "M_minus_B":grouped_pairs(predictions["B"],predictions["M"])}\n    general = {v:read_json(ROOT/f"evidence/general/{v}.json") for v in ("B","M","Q")}\n    challenges = {v:read_jsonl(ROOT/f"evidence/predictions/challenge-{v}.jsonl") for v in ("B","M","Q")}\n    challenge_ids = [digest(row["messages"]) for row in rows("challenge")]\n    if any([x["id"] for x in results] != challenge_ids for results in challenges.values()):\n        raise ValueError("Challenge IDs differ")\n    new_critical = [q["id"] for b,q in zip(challenges["B"],challenges["Q"])\n                    if b["complete"] and (not q["tool_selection"] or not q["schema_ok"] or not q["arguments_ok"])]\n    stage_paths = sorted((ROOT/"evidence/stages").glob("*.json"))\n    stage_reports = [read_json(p) for p in stage_paths if not re.search(r"\\.previous-",p.name)]\n    resource = [r.get("resources",{}) for r in stage_reports]\n    gpu_reports = [r for r in stage_reports if r.get("hardware")]\n    available = min((r["hardware"]["total_bytes"] for r in stage_reports if r.get("hardware")),default=0)\n    allocated = max((r.get("peak_allocated_bytes",0) for r in resource),default=0)\n    reserved = max((r.get("peak_reserved_bytes",0) for r in resource),default=0)\n    device_sampled = max((r.get("peak_device_used_mib_sampled",0)*2**20 for r in resource),default=0)\n    memory = {"usable_device_bytes":available,"max_allocated_bytes":allocated,"max_reserved_bytes":reserved,\n              "max_device_used_bytes_sampled":device_sampled,"headroom_bytes":available-max(reserved,device_sampled),\n              "per_gpu_stage":{r["stage"]:{k:r["resources"][k] for k in\n                  ("elapsed_seconds","peak_allocated_bytes","peak_reserved_bytes","peak_device_used_mib_sampled",\n                   "device_samples","peak_rss_bytes_sampled")} for r in gpu_reports},\n              "sampling_limit":"Device samples can miss short spikes; allocator reservation and sampled driver memory are distinct instruments."}\n    estimate = read_json(ROOT/"evidence/memory-estimate-before.json")\n    memory["preflight_budget_bytes"] = estimate["training_memory_preflight_budget_gib"]*2**30\n    memory["peak_allocated_minus_parameter_state_floor_bytes"] = allocated-(estimate["base_fp16_gib"]+estimate["lora_fp32_parameter_gradient_adam_gib"])*2**30\n    artifacts_ok = all(check_bound_artifact(frozen["artifacts"][v],artifact_path(v)) for v in ("M","Q","Q0"))\n    freeze_ok = file_sha(ROOT/"run_policy.json") == frozen["policy_sha256"] and file_sha(ROOT/"data/test.jsonl") == frozen["test_hash"]\n    freeze_ok = freeze_ok and all(file_sha(ROOT/name) == sha for name,sha in frozen["policy"]["config_hashes"].items())\n    freeze_ok = freeze_ok and file_sha(ROOT/"data/challenge.jsonl") == frozen["challenge_hash"]\n    freeze_ok = freeze_ok and file_sha(ROOT/"evidence/references.jsonl") == frozen["references_hash"]\n    required_stage_labels = [f"train-{s}" for s in SETTINGS["seeds"]]\n    required_stage_labels += [f"evaluate-validation-A-{s}" for s in SETTINGS["seeds"]]\n    required_stage_labels += [f"evaluate-{split}-{variant}" for split in ("test","challenge") for variant in ("B","M","Q")]\n    required_stage_labels += ["prepare","selftest","freeze","select","fidelity","merge-M","merge-Q","merge-Q0",\n                              "preflight-dry-run","bench-train","selected-adapter-check","selected-safe-check","selected-adapter-diff"]\n    stage_ok = all((ROOT/f"evidence/stages/{label}.json").exists() and read_json(ROOT/f"evidence/stages/{label}.json")["status"] == "passed" for label in required_stage_labels)\n    if stage_ok:\n        stage_ok = benchmark_headroom(read_json(ROOT/"evidence/stages/bench-train.json"), SETTINGS["allocator_config"])["passed"]\n    resource_ok = bool(gpu_reports) and all(r["resources"]["device_samples"] > 0 and\n        not any("cuda_measurement_error" in segment for segment in r["resources"]["segments"]) for r in gpu_reports)\n    identities = {v:read_json(ROOT/f"evidence/identity-{v}.json") for v in ("M","Q")}\n    no_op_caught = next((x["Soup_status"] == "observed_caught" for x in coverage if x["failure"] == "no_op_adapter"),False)\n    integrity = {"required_stage_reports_pass":stage_ok,"policy_and_test_unchanged":freeze_ok,"artifact_hashes_bound":artifacts_ok,\n        "updated_and_active_adapter":training["observations"]["first_materialized_update"] is not None\n            and training["adapter_effect"]["max_reference_logprob_error"] > max(1e-5,10*frozen["calibration"]["floor"]),\n        "selected_frozen_weights_unchanged":training["frozen_projection_hashes_before"] == training["frozen_projection_hashes_after"],\n        "disabled_base_agreement":training["disabled_vs_pristine"]["max_reference_logprob_error"] <= max(.002,10*frozen["calibration"]["floor"]),\n        "merge_fidelity":fidelity["merge_faithful"],"scorer_data_controls":read_json(ROOT/required[2])["all_passed"],\n        "base_for_merge_control":fidelity["controls"]["base_substituted_for_merged_rejected"],\n        "q0_for_q_control":fidelity["controls"]["q0_substituted_for_q_identity_rejected"] and fidelity["controls"]["q_distinguishable_from_q0"],\n        "Soup_noop_adapter_control":no_op_caught,"portable_M_Q":all(identities[v]["portable_base_adapter_reads_blocked"] for v in identities),\n        "real_saved_nf4":identities["Q"]["quantized_linear_count"] == 196,\n        "no_new_critical_challenge_failures":not new_critical,\n        "one_gib_tested_headroom":resource_ok and available > 0 and memory["headroom_bytes"] >= 2**30,\n        "three_seed_development_complete":len(SETTINGS["seeds"]) >= 3,\n        "Soup_environment_bounds_ok":(ROOT/"evidence/stages/env-check.json").exists()\n            and read_json(ROOT/"evidence/stages/env-check.json").get("result",{}).get("exit_code") == 0,\n        "Soup_fixture_coverage_resolved":all(x["Soup_status"] in {"observed_caught","observed_missed"} for x in coverage)}\n    verdict = decision_gates(pairs["Q_minus_B"],pairs["Q_minus_M"],general["B"]["scores"],general["Q"]["scores"],integrity)\n    verdict.update(utc=now(),selected_seed=selected,metrics={v:summarize_predictions(x) for v,x in predictions.items()},\n        paired_bootstrap=pairs,general_suites={v:g["scores"] for v,g in general.items()},memory=memory,\n        new_critical_challenges=new_critical,selection=frozen["selection"],policy_sha256=frozen["policy_sha256"],\n        scope_limits=["Four validation distribution probes and four weight projections do not prove full-distribution equivalence.",\n                      "Prepared holdouts are source-shifted toward primary Hermes; source labels are absent from the four-file export.",\n                      "References are structurally audited; semantic correctness and alternative valid calls are not automatically adjudicated.",\n                      "Bundled general suites and six challenge turns have finite coverage; this is not an arbitrary-tool safety claim."],\n        remediation="Inspect exact failed gates: export/identity failures need re-export or verifier fixes; task/regression failures need development work before a fresh test policy.")\n    write_json(ROOT/"verdict.json",verdict)\n    write_json(ROOT/"evidence/memory-reconciliation.json",memory)\n    for variant in ("M","Q"):\n        payload = {"task":{"mode":"metric","base":verdict["metrics"]["B"]["complete"],"tuned":verdict["metrics"][variant]["complete"]},\n                   "benchmarks":{name:{"base":general["B"]["scores"][name],"tuned":general[variant]["scores"][name]} for name in general["B"]["scores"]},\n                   "provenance":{"source":"offline replay of measured predictions","runtime_sha256":file_sha(__file__),\n                                 "predictions_sha256":file_sha(ROOT/f"evidence/predictions/test-{variant}.jsonl"),"artifact_sha256":frozen["artifacts"][variant]["sha256"]}}\n        destination = ROOT/f"evidence/soup-ship-input-{variant}.json"\n        write_json(destination,payload)\n        run_soup(["ship","--evidence",str(destination),"--forgetting-threshold","0.05","--output",str(ROOT/f"evidence/soup-ship-{variant}.json")],allowed=(0,2))\n    # Reuse quant-check on cached, already generated single-call wrappers; never pretend this loaded models.\n    from soup_cli.eval.custom import EvalTask\n    from soup_cli.eval.quant_check import run_quant_check\n    tasks = [EvalTask(prompt=x["id"],expected=canonical({"function":x["expected"][0]}),scoring="tool_call_match") for x in refs if x["call_count"] == 1]\n    indexes = {v:{x["id"]:x for x in predictions[v]} for v in ("M","Q")}\n    def cached(variant,prompt):\n        row = indexes[variant][prompt]\n        try:\n            calls = parse_calls(row["text"])\n            return canonical({"function":calls[0]}) if len(calls)==1 and not row["truncated"] else "INVALID"\n        except ValueError:\n            return "INVALID"\n    quant_result = run_quant_check(before_gen=lambda p:cached("M",p),after_gen=lambda p:cached("Q",p),tasks=tasks,task_name="single-call subset, cached measured generations")\n    write_json(ROOT/"evidence/soup-quant-check.json",{"source":"cached-generation decision replay; no additional model load","single_call_turns":len(tasks),"result":json.loads(quant_result.to_json())})\n    return {"outputs":["verdict.json","evidence/memory-reconciliation.json","evidence/silent-failure-coverage.json"],"verdict":verdict["verdict"],"blockers":verdict["blockers"]}\n\n\ndef report_pages(verdict):\n    metrics = verdict.get("metrics") or {}\n    page1 = ["SOUP / HERMES / QWEN2.5-1.5B", "Measured experiment report - " + verdict["verdict"], "",\n             f"Base: {MODEL_ID}@{MODEL_REV[:12]}",f"Soup: {SOUP_REV[:12]}; prepared Kaggle dataset: pernavjain/tool-calling",\n             "Data: 1600 training /150 validation /150 test; exact file hashes in run_policy.json.",\n             "Training: frozen FP16 base, rank8 all seven projections, batch1, accumulation8, two epochs.",\n             "Recipe: LR1e-4, cosine decay, 5% warmup; export the lowest validation-loss epoch.",\n             "Seed selection uses validation complete-turn correctness, then NLL, then seed.", "",\n             "TASK RESULTS - complete ordered turns; every held-out turn remains in denominator"]\n    for variant in ("B","M","Q"):\n        value = metrics.get(variant)\n        page1.append(f"{variant}: " + (f"{value[\'complete\']:.3%} complete; {value[\'parse_ok\']:.3%} parse; assistant NLL={value[\'mean_assistant_nll\']:.5f}; n={value[\'n\']}" if value else "NOT MEASURED"))\n    page1 += ["", "PAIRED GROUP BOOTSTRAP - 10000 seeded resamples"]\n    for name,pair in verdict.get("paired_bootstrap",{}).items():\n        page1.append(f"{name}: {pair[\'delta\']:+.3%};95% CI [{pair[\'ci95\'][0]:+.3%},{pair[\'ci95\'][1]:+.3%}];groups={pair[\'groups\']}")\n    memory = verdict.get("memory",{})\n    page1 += ["", "RESOURCE RECONCILIATION"]\n    for name in ("usable_device_bytes","max_allocated_bytes","max_reserved_bytes","max_device_used_bytes_sampled","headroom_bytes"):\n        page1.append(f"{name}: " + (f"{memory[name]/2**30:.3f} GiB" if name in memory else "NOT MEASURED"))\n    page1 += ["Allocator counters belong to the tensor-owning child. Device/RSS samples can miss spikes.",\n              "The original estimate precedes optimization. Activations, workspaces and context add to the floor.", ""]\n    if metrics:\n        page1 += ["ARTIFACTS: adapter, merged-fp16, merged-nf4 retained under artifacts/.",\n                  "Cold M/Q loads use local files, block base/adapter reads and reject CPU/meta fallback."]\n    else:\n        page1.append("ARTIFACTS: verification incomplete. See stage reports for observed outputs.")\n    page2 = ["GATES, CONTROLS AND LIMITATIONS", "", "RELEASE GATES"]\n    page2 += [f"{\'PASS\' if passed is True else \'BLOCK\'}: {name}" for name,passed in verdict.get("gates",{}).items()]\n    page2 += ["", "GENERAL REGRESSION - absolute suite scores B /M /Q"]\n    general = verdict.get("general_suites",{})\n    for name in general.get("B",{}):\n        page2.append(name + ": " + " / ".join(f"{general[v][name]:.3f}" for v in ("B","M","Q")))\n    page2 += ["", "LIMITS AND REFLECTION"]\n    page2 += verdict.get("scope_limits",["Required experiment results missing; inspect stage errors and verdict blockers."])\n    reflection = SETTINGS.get("reflection","").strip()\n    page2.append("Author reflection: " + (reflection or "PENDING - add your own observed surprise, concern and next step before submission."))\n    page2.append("AI disclosure: AI assisted notebook/code design; this report uses saved run outputs. Author should describe their checks and edits.")\n    page2.append("This is an automated report draft. Inspect the final two-page PDF and complete reflection before submission.")\n    return page1,page2\n\n\ndef package_stage():\n    import textwrap\n    from reportlab.pdfgen import canvas\n    from reportlab.lib.pagesizes import A4\n    from pypdf import PdfReader\n    verdict = read_json(ROOT/"verdict.json") if (ROOT/"verdict.json").exists() else {"verdict":"DON\'T SHIP","metrics":None,"gates":{"required_verdict_present":False}}\n    pages = report_pages(verdict)\n    markdown = "\\n\\n---\\n\\n".join("\\n".join(page) for page in pages)\n    (ROOT/"REPORT.md").write_text(markdown+"\\n")\n    pdf_path = ROOT/"report.pdf"\n    pdf = canvas.Canvas(str(pdf_path),pagesize=A4)\n    width,height = A4\n    for number,paragraphs in enumerate(pages,1):\n        pdf.setFont("Helvetica-Bold",13)\n        pdf.drawString(42,height-40,"Soup AI Engineer experiment - " + ("results" if number == 1 else "verification"))\n        lines = []\n        for paragraph in paragraphs:\n            ascii_text = paragraph.encode("ascii","replace").decode()\n            lines.extend(textwrap.wrap(ascii_text,width=96) or [""])\n        if len(lines) > 61:\n            raise ValueError("Report would overflow two pages; shorten the author reflection/report text")\n        text = pdf.beginText(42,height-65)\n        text.setFont("Courier",8)\n        text.setLeading(11)\n        for line in lines:\n            text.textLine(line)\n        pdf.drawText(text)\n        pdf.setFont("Helvetica",8)\n        pdf.drawString(42,30,f"{number}/2 - measured evidence: evidence/; verdict: verdict.json")\n        pdf.showPage()\n    pdf.save()\n    assert len(PdfReader(pdf_path).pages) == 2\n    (ROOT/"REFLECTION_AND_AI_USE.md").write_text(\n        "# Author reflection and AI-use disclosure\\n\\n" + (SETTINGS.get("reflection","").strip() or\n        "Add your first-person reflection based on this run: what surprised you, what concerns you, and what you would test next.") +\n        "\\n\\nAI assisted the notebook and verifier implementation. Add which suggestions you accepted, changed or rejected and how you independently checked them.\\n")\n    outputs = ["REPORT.md","report.pdf","REFLECTION_AND_AI_USE.md"]\n    return {"outputs":outputs,"verdict":verdict["verdict"],"pdf_pages":2,\n            "report_status":"automated draft; human reflection/visual inspection pending"}\n\n\ndef main():\n    global ROOT,SETTINGS,CAPTURE\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root",required=True,help="Kaggle run directory with settings.json, data/ and artifacts/")\n    sub = parser.add_subparsers(dest="stage",required=True)\n    for name in ("prepare","select","fidelity","selftest","freeze","finalize","package"):\n        sub.add_parser(name)\n    p = sub.add_parser("train");p.add_argument("--seed",type=int,required=True)\n    p = sub.add_parser("merge");p.add_argument("--variant",choices=("M","Q","Q0"),required=True)\n    p = sub.add_parser("evaluate")\n    p.add_argument("--variant",choices=("B","A","A0","M","Q","Q0"),required=True)\n    p.add_argument("--split",choices=("validation","test","challenge"),required=True)\n    p.add_argument("--seed",type=int)\n    p.add_argument("--general",action="store_true")\n    p.add_argument("--probes-only",action="store_true")\n    p = sub.add_parser("cli")\n    p.add_argument("--label",required=True)\n    p.add_argument("--allowed",default="0")\n    p.add_argument("--cuda-owner",action="store_true")\n    p.add_argument("arguments",nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n    ROOT = Path(args.root).resolve()\n    SETTINGS = read_json(ROOT/"settings.json")\n    os.chdir(ROOT)\n    os.environ["CUDA_VISIBLE_DEVICES"] = SETTINGS["gpu_uuid"]\n    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"\n    allocator_config = SETTINGS.setdefault("allocator_config", T4_ALLOCATOR_CONFIG)\n    # Set the alias to the same value for older PyTorch versions.\n    os.environ["PYTORCH_ALLOC_CONF"] = allocator_config\n    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = allocator_config\n    os.environ["TOKENIZERS_PARALLELISM"] = "false"\n    os.environ["WANDB_MODE"] = "disabled"\n    os.environ["OMP_NUM_THREADS"] = "2"\n    os.environ["MKL_NUM_THREADS"] = "2"\n    if args.stage != "prepare":\n        os.environ["HF_HUB_OFFLINE"] = "1"\n        os.environ["TRANSFORMERS_OFFLINE"] = "1"\n    if args.stage in {"train","evaluate","merge","freeze","selftest"}:\n        for name,sha in EXPECTED_DATA.items():\n            local = ROOT/("data/"+name if name.endswith(".jsonl") else name)\n            if file_sha(local) != sha:\n                raise ValueError("Prepared input bytes changed: " + name)\n        policy = read_json(ROOT/"run_policy.json")\n        if policy.get("allocator_config") != SETTINGS["allocator_config"]:\n            raise ValueError("Frozen allocator policy changed")\n        if policy.get("trainer_policy") != TRAINER_POLICY:\n            raise ValueError("Frozen trainer policy changed; use a new run for this training recipe")\n        if any(file_sha(ROOT/name) != sha for name,sha in policy["config_hashes"].items()):\n            raise ValueError("Frozen training configuration changed")\n    if args.stage in {"train","evaluate"} and getattr(args,"variant",None) in {"A","A0"} and args.seed is None:\n        parser.error("Adapter evaluation needs --seed")\n    if args.stage == "evaluate" and args.probes_only and args.split != "validation":\n        parser.error("Probe-only runs use the fixed validation probes")\n    label = args.stage\n    if args.stage == "cli":label = args.label\n    elif args.stage == "train":label = f"train-{args.seed}"\n    elif args.stage == "merge":label = f"merge-{args.variant}"\n    elif args.stage == "evaluate":label = f"evaluate-{args.split}-{args.variant}" + (f"-{args.seed}" if args.seed is not None else "")\n    if not re.fullmatch(r"[A-Za-z0-9_.-]+",label):\n        parser.error("Invalid stage label")\n    stage_path = ROOT/f"evidence/stages/{label}.json"\n    stage_path.parent.mkdir(parents=True,exist_ok=True)\n    request = {key:value for key,value in vars(args).items() if key != "root"}\n    runtime_sha = file_sha(__file__)\n    if stage_path.exists():\n        old = read_json(stage_path)\n        if old.get("status") == "passed" and args.stage not in {"finalize","package"}:\n            if old.get("runtime_sha256") != runtime_sha or old.get("settings_sha256") != digest(SETTINGS) or old.get("request") != request:\n                raise ValueError("Cached stage settings/code/request drifted; create a new run")\n            for name,sha in old.get("output_bindings",{}).items():\n                if not (ROOT/name).is_file() or file_sha(ROOT/name) != sha:\n                    raise ValueError("Cached stage output changed: " + name)\n            print("Verified completed stage; reusing: " + label,flush=True)\n            return\n        stage_path.rename(stage_path.with_name(label+".previous-"+str(time.time_ns())+".json"))\n    report = {"stage":label,"request":request,"started_utc":now(),"status":"running","runtime_sha256":runtime_sha}\n    write_json(stage_path,report)\n    faulthandler.enable()\n    CAPTURE = ResourceCapture(label)\n    CAPTURE.start()\n    try:\n        if args.stage == "prepare":result = prepare_stage()\n        elif args.stage == "train":result = train_stage(args.seed)\n        elif args.stage == "evaluate":result = evaluate_stage(args.variant,args.split,args.seed,args.general,args.probes_only)\n        elif args.stage == "merge":result = merge_stage(args.variant)\n        elif args.stage == "select":result = select_stage()\n        elif args.stage == "fidelity":result = fidelity_stage()\n        elif args.stage == "selftest":result = selftest_stage()\n        elif args.stage == "freeze":result = freeze_stage()\n        elif args.stage == "finalize":result = finalize_stage()\n        elif args.stage == "package":result = package_stage()\n        else:\n            if args.cuda_owner:cuda()\n            arguments = args.arguments[1:] if args.arguments[:1] == ["--"] else args.arguments\n            code = run_soup(arguments,allowed=tuple(int(x) for x in args.allowed.split(",")))\n            outputs = []\n            for flag in ("--output","--state"):\n                if flag in arguments:\n                    target = (ROOT/arguments[arguments.index(flag)+1]).resolve()\n                    if target.is_file():\n                        outputs.append(str(target.relative_to(ROOT)))\n            result = {"exit_code":code,"arguments":arguments,"outputs":outputs}\n        report.update(status="passed",result=result)\n        report["output_bindings"] = {name:file_sha(ROOT/name) for name in result.get("outputs",[]) if (ROOT/name).is_file()}\n    except BaseException as exc:\n        report.update(status="failed",error=f"{type(exc).__name__}: {exc}")\n        traceback.print_exc()\n        raise\n    finally:\n        report["resources"] = CAPTURE.finish()\n        if TORCH is not None and TORCH.cuda.is_initialized():\n            props = TORCH.cuda.get_device_properties(0)\n            report["hardware"] = {"name":props.name,"total_bytes":props.total_memory,"visible_count":TORCH.cuda.device_count(),\n                                  "selected_physical_uuid":SETTINGS["gpu_uuid"],"capability":list(TORCH.cuda.get_device_capability(0)),\n                                  "torch_version":TORCH.__version__,"torch_cuda_build":TORCH.version.cuda,\n                                  "allocator_backend":TORCH.cuda.get_allocator_backend(),\n                                  "allocator_config":os.environ["PYTORCH_ALLOC_CONF"]}\n        if label == "bench-train":\n            report["memory_gate"] = benchmark_headroom(report, SETTINGS["allocator_config"])\n        report.update(finished_utc=now(),settings_sha256=digest(SETTINGS))\n        write_json(stage_path,clean_observation(report))\n        if args.stage == "evaluate" and args.variant in {"M","Q"}:\n            shutil.rmtree(ROOT/f"artifacts/portable/{args.variant}",ignore_errors=True)\n        event("stage_end",stage=label,status=report["status"])\n\n\nif __name__ == "__main__":\n    main()\n'
(RUN/'verify_artifacts.py').write_text(RUNTIME_SOURCE)
print('Verifier SHA-256:',hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest())

Verifier SHA-256: f3cb5f8ac750e2cdf86f9cc236f9f327a957c56e8e937d387ac589e45cd4bbb1


In [4]:
def stage(command,*arguments,label=None,cuda_owner=False,allowed=(0,),timeout=None):
    argv = [sys.executable,'-u',str(RUN/'verify_artifacts.py'),'--root',str(RUN),command]
    if command == 'cli':
        argv += ['--label',label,'--allowed',','.join(map(str,allowed))]
        if cuda_owner:
            argv += ['--cuda-owner']
        argv += ['--',*map(str,arguments)]
    else:
        argv += list(map(str,arguments))
    return external(argv,label or command,timeout=timeout or STAGE_TIMEOUT_SECONDS)

def soup(label,*arguments,allowed=(0,),heavy=False,timeout=600):
    return stage('cli',*arguments,label=label,cuda_owner=heavy,allowed=allowed,timeout=timeout)

def show(relative):
    print(json.dumps(json.loads((RUN/relative).read_text()),indent=2))

## Phase3 · Identity, exact masks and the estimate BEFORE training

The four Kaggle files must match the prepared hashes. No split is changed.
References/tools are reconstructed from the stored assistant/system messages.
Explicit validation loading is retained with `val_split:0`.

Every prepared row receives an exact response/EOT mask check using Soup's
formatter. The collator is tested with normal padding and padding equal
to EOS. Cross-split normalized requests and tool-family overlaps are rejected.
This reconstructs structural evidence, not missing source-category labels.
The known primary/Glaive source shift remains a limitation of this dataset.

Only this stage uses the network to fetch the pinned base. Later child
stages use local paths with Hugging Face offline mode enabled.

In [5]:
stage('prepare',timeout=1800)
show('evidence/data-ready.json')
show('evidence/memory-estimate-before.json')
show('evidence/training-plan.json')

prepare started. Log: evidence/logs/prepare-1794285635954143612.log
prepare: exit=0; log=evidence/logs/prepare-1794285635954143612.log
{
  "counts": {
    "train": 1600,
    "validation": 150,
    "test": 150
  },
  "max_length": 1024,
  "generation_cap": 320,
  "all_reference_masks_exact": true,
  "reserved_token_check": true,
  "split_overlap": 0,
  "collator_padding_checks": true,
  "native_template_sha256": "2e2d2512cfe46af53dc1eed45368ecaab26ac4461c480e6b69fe571cf0ceaa75",
  "training_template_sha256": "9f7753045eff11c1260c847986095b0518eaa7e748811aeb5f1a5bc66dde5cd3",
  "group_counts": {
    "train": 402,
    "validation": 150,
    "test": 150
  }
}
{
  "created_before_training": "2026-11-10T04:41:30.545859+00:00",
  "unique_parameters": 1543714304,
  "lora_parameters_expected": 9232384,
  "base_fp16_gib": 2.875391960144043,
  "lora_fp32_parameter_gradient_adam_gib": 0.1375732421875,
  "one_fp16_full_logits_gib": 0.289794921875,
  "one_fp32_full_logits_gib": 0.57958984375,
  "kv_

## Phase3 · Soup pre-flight and full-split data reports

All commands retain actual exit codes and raw stdout/stderr. Validation/test
omit `--show-mask`; passing zero is invalid in the pinned Soup version.
Each doctor report must scan the complete split and return OK. The `expect`
token check is only a coarse whitespace/nonempty sanity check; exact Qwen
token counts above enforce the context limit.

Doctor's optional-package warnings are advisory. `env check` exit3 is retained
and blocks a final SHIP if Soup's declared dependency bounds/lock are not clean.
`plan` and `profile` produce estimates; they are never labelled measured peaks.

In [6]:
first_seed = SEEDS[0]
config = f'configs/seed-{first_seed}.yaml'
for label,args in (
    ('help-data-doctor',['data','doctor','--help']),('help-bench-train',['bench','train','--help']),
    ('help-merge',['merge','--help']),('help-ship',['ship','--help'])):
    soup(label,*args)
soup('doctor','doctor','--config',config)
soup('env-lock','env','lock','--output','evidence/soup-env.lock')
soup('env-check','env','check','--lock','evidence/soup-env.lock',allowed=(0,3))
soup('data-inspect','data','inspect','data/train.jsonl','--rows','2')
soup('data-stats','data','stats','data/train.jsonl')
expected_counts = {'train':1600,'validation':150,'test':150}
maximum = json.loads((RUN/'settings.json').read_text())['max_length']
for split,count in expected_counts.items():
    soup(f'validate-{split}','data','validate',f'data/{split}.jsonl','--format','chatml','--min-valid-fraction','1')
    args = ['data','doctor',f'data/{split}.jsonl','--model','artifacts/base','--format','chatml',
            '--max-length',str(maximum),'--sample',str(count),'--train-on-responses-only','--train-on-eot',
            '--output',f'evidence/data-doctor-{split}.json']
    if split == 'train':
        args += ['--show-mask','5']
    soup(f'doctor-{split}',*args)
    report = json.loads((RUN/f'evidence/data-doctor-{split}.json').read_text())
    assert report['rows_scanned'] == count and report['overall'] == 'OK'
(RUN/'coarse-expectations.yaml').write_text(
    'expectations:\n'
    '  - name: expect_token_length_between\n'
    '    args:\n'
    '      min_tokens: 1\n'
    '      max_tokens: 100000\n')
soup('expect-coarse','expect','data/train.jsonl','coarse-expectations.yaml')
soup('profile','profile','--config',config,'--json')
soup('plan','plan','--config',config,'--state','evidence/soup.tfstate')
soup('preflight-dry-run','train','--config',config,'--dry-run')

help-data-doctor started. Log: evidence/logs/help-data-doctor-1794285705062508958.log
help-data-doctor: exit=0; log=evidence/logs/help-data-doctor-1794285705062508958.log
help-bench-train started. Log: evidence/logs/help-bench-train-1794285705980952296.log
help-bench-train: exit=0; log=evidence/logs/help-bench-train-1794285705980952296.log
help-merge started. Log: evidence/logs/help-merge-1794285706848803868.log
help-merge: exit=0; log=evidence/logs/help-merge-1794285706848803868.log
help-ship started. Log: evidence/logs/help-ship-1794285707716171425.log
help-ship: exit=0; log=evidence/logs/help-ship-1794285707716171425.log
doctor started. Log: evidence/logs/doctor-1794285708583655588.log
doctor: exit=0; log=evidence/logs/doctor-1794285708583655588.log
env-lock started. Log: evidence/logs/env-lock-1794285723685539179.log
env-lock: exit=0; log=evidence/logs/env-lock-1794285723685539179.log
env-check started. Log: evidence/logs/env-check-1794285724502883153.log
env-check: exit=0; log=evi

0

## Phase3 · Deliberate data/scorer failures and scratch training benchmark

Correct JSON is not sufficient evidence of correct tool selection/arguments.
These fixtures exercise wrong values, boolean/number confusion, extra calls,
malformed payloads, absent tools, double ChatML and truncation. The verifier
uses Soup's `tool_call_match` on aligned wrapped calls, plus outer strict
parsing/schema/completeness checks. Soup catches/misses are recorded honestly.

The20-step Soup benchmark uses scratch output. Its built-in allocator reset
excludes setup; the primary run below measures load and optimizer phases
independently. The benchmark's adapter/optimizer is never reused for training.

The worker requests expandable segments and cache reclamation with PyTorch's
native CUDA allocator. Variable sequence lengths can leave cached blocks that
cannot be reused. The gate uses the larger of sampled device usage and peak
allocator reservation, and requires at least 1 GiB of headroom. Read the
allocated, reserved and device peaks if the gate stops the run.

In [7]:
stage('selftest')
soup('fixture-malformed-validate','data','validate','controls/malformed_call.jsonl',
     '--format','chatml','--min-valid-fraction','1',allowed=(0,1,2,3))
soup('fixture-double-validate','data','validate','controls/double_template.jsonl',
     '--format','chatml','--min-valid-fraction','1',allowed=(0,1,2,3))
for label,filename,length in [('fixture-missing-doctor','missing_tools',maximum),
                              ('fixture-length-doctor','length_boundary',16)]:
    soup(label,'data','doctor',f'controls/{filename}.jsonl','--model','artifacts/base',
         '--format','chatml','--max-length',str(length),'--sample','1','--train-on-eot',
         '--output',f'evidence/{label}.json',allowed=(0,1,2,3))
soup('bench-train','bench','train','--config',config,'--steps','20','--warmup','3',
     '--output','evidence/bench-train.json',heavy=True,timeout=1800)
bench_stage = json.loads((RUN/'evidence/stages/bench-train.json').read_text())
memory_gate = bench_stage['memory_gate']
print(json.dumps(memory_gate, indent=2))
if not memory_gate['passed']:
    raise RuntimeError(f'Scratch T4 memory gate failed. See {RUN / "evidence/stages/bench-train.json"} '
                       'and the device trace before starting primary training.')

selftest started. Log: evidence/logs/selftest-1794285772030299789.log
selftest: exit=0; log=evidence/logs/selftest-1794285772030299789.log
fixture-malformed-validate started. Log: evidence/logs/fixture-malformed-validate-1794285782823275570.log
fixture-malformed-validate: exit=0; log=evidence/logs/fixture-malformed-validate-1794285782823275570.log
fixture-double-validate started. Log: evidence/logs/fixture-double-validate-1794285783692295837.log
fixture-double-validate: exit=0; log=evidence/logs/fixture-double-validate-1794285783692295837.log
fixture-missing-doctor started. Log: evidence/logs/fixture-missing-doctor-1794285784559776988.log
fixture-missing-doctor: exit=0; log=evidence/logs/fixture-missing-doctor-1794285784559776988.log
fixture-length-doctor started. Log: evidence/logs/fixture-length-doctor-1794285794048377260.log
fixture-length-doctor: exit=0; log=evidence/logs/fixture-length-doctor-1794285794048377260.log
bench-train started. Log: evidence/logs/bench-train-1794285803587

## Phase4 · Base development measurements and fresh training seeds

Base validation predictions are recorded first. Each seed then starts from
the pristine pinned base and a fresh LoRA/optimizer. Train for **two epochs**
(400 optimizer steps per seed for1600 rows/effective batch8), at **1e-4** LR
with **5% warmup**, cosine decay, rank8/alpha16/dropout0.05 and AdamW.
This is a reasoned recipe, not a claim that the hyperparameters are optimal.

Evaluate and save at each epoch. Reload the checkpoint with the lowest
validation loss, and verify that the exported adapter tensors exactly match
it. `evidence/training-<seed>.json` records both epoch losses and the selected
epoch/step. Across seeds, select by validation function-call accuracy, then
token-weighted NLL. A selected checkpoint may come from epoch1 if epoch2
performs worse. Training always completes both epochs.

Evaluation uses batch1 and returns loss only, avoiding retained prediction
logits. Training disables KV caching and uses non-reentrant gradient
checkpointing. The first seed measures
three baseline probe repeats and the initialized adapter's numerical floor
**before** any update. The tolerance formula was frozen in the run policy.

Training evidence includes optimizer membership, shifted supervised tokens,
AMP skips, the first materialized nonzero-LR update, changed finite adapters,
unchanged selected frozen projections, adapter on/off effects and real logs.
Zero-LR warmup steps are not counted as useful updates. First forward,
backward, optimizer, evaluation and saves retain allocator segment peaks.

Validation accuracy selects the final seed, with token-weighted NLL and seed
as deterministic tie-breaks. Test outcomes cannot influence selection.
Collector timing is synchronized/instrumented and is labelled accordingly.

The benchmark field `trainable_parameters:392` counts LoRA tensors. The
actual trainable weight count is9,232,384, recorded separately in training
and checkpoint-selection evidence. Training is FP16 LoRA; NF4 is Phase5.

Failures include the specific seed, child exit code and the final log lines
in the exception, and save `evidence/last-failure.json`. The former periodic
five-minute `Timeout` stack dump has been removed: it was a diagnostic
timer, not a training deadline. The real subprocess timeout is configured
by `STAGE_TIMEOUT_SECONDS` (four hours per stage by default).

In [8]:
stage('evaluate','--variant','B','--split','validation',label='evaluate-validation-B')
for seed in SEEDS:
    stage('train','--seed',seed,label=f'train-{seed}')
    training = json.loads((RUN/f'evidence/training-{seed}.json').read_text())
    print(json.dumps({'seed':seed,**training['checkpoint_selection']},indent=2))
    stage('evaluate','--variant','A','--seed',seed,'--split','validation',label=f'evaluate-validation-A-{seed}')
stage('select')
selection = json.loads((RUN/'evidence/selection.json').read_text())
selected_seed = selection['selected']['seed']
show('evidence/selection.json')

evaluate-validation-B started. Log: evidence/logs/evaluate-validation-B-1794285936101879720.log
evaluate-validation-B: exit=0; log=evidence/logs/evaluate-validation-B-1794285936101879720.log
train-42 started. Log: evidence/logs/train-42-1794286602310377692.log
train-42: exit=0; log=evidence/logs/train-42-1794286602310377692.log
{
  "seed": 42,
  "metric": "eval_loss",
  "selection_uses_test": false,
  "epoch_validation": [
    {
      "epoch": 1.0,
      "step": 200,
      "eval_loss": 0.04504300281405449
    },
    {
      "epoch": 2.0,
      "step": 400,
      "eval_loss": 0.04345468059182167
    }
  ],
  "epochs_completed": 2.0,
  "optimizer_steps_completed": 400,
  "selected_epoch": 2.0,
  "selected_step": 400,
  "best_validation_loss": 0.04345468059182167,
  "checkpoint": "artifacts/training-42/checkpoint-400",
  "checkpoint_adapter_sha256": "d6c2a6fce66168a8831c326aa02096471eae7f3ed86df5fa43210905325d9b0f",
  "export_matches_checkpoint_tensors": true,
  "trainable_tensor_count": 

## Phase4 · Soup adapter liveness, serialization and effective-update diff

Training checkpoints/optimizer states stay under `artifacts/training-SEED/`.
The deployable `adapter-SEED/` contains a clean safe serialization. This avoids
mistaking the Trainer's own optimizer pickle files for deployable weights
during Soup's recursive strict-safetensors check.

The initialized no-op adapter should be rejected by adapter liveness checking.
Its actual exit is retained; an unexpected pass cannot silently validate the
control. Alive trained layers establish activity, not useful learning.

In [9]:
soup('initial-adapter-check','adapters','check',f'artifacts/initial-{SEEDS[0]}','--json',allowed=(0,2))
soup('selected-adapter-check','adapters','check',f'artifacts/adapter-{selected_seed}','--json')
soup('selected-safe-check','adapters','check-safetensors',f'artifacts/adapter-{selected_seed}','--strict')
soup('selected-adapter-diff','adapters','diff',f'artifacts/initial-{selected_seed}',
     f'artifacts/adapter-{selected_seed}','--format','json','--output','evidence/adapter-diff.json')

initial-adapter-check started. Log: evidence/logs/initial-adapter-check-1794296086929234836.log
initial-adapter-check: exit=0; log=evidence/logs/initial-adapter-check-1794296086929234836.log
selected-adapter-check started. Log: evidence/logs/selected-adapter-check-1794296087898153574.log
selected-adapter-check: exit=0; log=evidence/logs/selected-adapter-check-1794296087898153574.log
selected-safe-check started. Log: evidence/logs/selected-safe-check-1794296088816262863.log
selected-safe-check: exit=0; log=evidence/logs/selected-safe-check-1794296088816262863.log
selected-adapter-diff started. Log: evidence/logs/selected-adapter-diff-1794296089683519657.log
selected-adapter-diff: exit=0; log=evidence/logs/selected-adapter-diff-1794296089683519657.log


0

## Phase5 · Save dense M, standalone NF4 Q and quantized-base control Q0

These are separate tensor-owning processes with child-local allocator and
sampled host/device peaks. Soup's NF4 save path repeats a dense CPU merge
internally; that cost is retained. Tied embeddings remain floating point,
so neither file size nor runtime memory should be assumed to shrink4x.

The default retains all actual model outputs. It does not delete M/Q after a
smoke reload. Dense staging needs several GiB of host RAM and free disk.
`4bit_forced` is deliberately avoided for this tied-embedding model.

In [10]:
for variant in ('M','Q','Q0'):
    stage('merge','--variant',variant,timeout=1800)

merge started. Log: evidence/logs/merge-1794296091158798056.log
merge: exit=0; log=evidence/logs/merge-1794296091158798056.log
merge started. Log: evidence/logs/merge-1794296143179296814.log
merge: exit=0; log=evidence/logs/merge-1794296143179296814.log
merge started. Log: evidence/logs/merge-1794296189386577465.log
merge: exit=0; log=evidence/logs/merge-1794296189386577465.log


## Phase5/6 · Cold reload, distribution/weight fidelity and wrong-export controls

M/Q are loaded from temporary standalone hardlinks with offline mode,
local-only loading and an audit hook that rejects base/adapter file reads.
Those temporary links are removed after verification; original exports remain.
Q must contain196 actual NF4 linear modules, nested quantization state and
FP16 compute. Passing a fresh quantization config at reload is forbidden.

Probe comparisons retain reference log probabilities, assistant NLL and
float32 KL at bounded validation positions. Four projections also compare
merged weights against base plus the scaled low-rank update. B substituted
for M and Q0 substituted for Q must be rejected, or shipment is blocked.
Quantization-only B/Q0 distortion is reported separately from M/Q.

In [11]:
for variant in ('M','Q'):
    stage('evaluate','--variant',variant,'--split','validation')
stage('evaluate','--variant','Q0','--split','validation','--probes-only')
stage('fidelity')
show('evidence/fidelity-and-export-controls.json')

evaluate started. Log: evidence/logs/evaluate-1794296236895725240.log
evaluate: exit=0; log=evidence/logs/evaluate-1794296236895725240.log
evaluate started. Log: evidence/logs/evaluate-1794296738302888339.log
evaluate: exit=0; log=evidence/logs/evaluate-1794296738302888339.log
evaluate started. Log: evidence/logs/evaluate-1794297493441174336.log
evaluate: exit=0; log=evidence/logs/evaluate-1794297493441174336.log
fidelity started. Log: evidence/logs/fidelity-1794297511750590683.log
fidelity: exit=0; log=evidence/logs/fidelity-1794297511750590683.log
{
  "calibration": {
    "utc_before_training": "2026-11-10T04:57:21.420031+00:00",
    "repeat_base_runs": 3,
    "repeat_error": 0.0,
    "initialized_adapter_error": 0.0,
    "floor": 0.0,
    "merge_reference_logprob_tolerance": 0.02,
    "ceiling": 0.1,
    "scope": "four fixed validation rows; deterministic baseline under actual kernels"
  },
  "A_to_M": {
    "max_reference_logprob_error": 0.03867495059967041,
    "mean_reference_log

## Phase6 · Freeze policy and unseal final held-out evaluation

The policy requires Q to gain at least5 percentage points over B, a positive
paired95% interval, at most2 points loss versus M, a Q-M lower confidence
bound no worse than-5 points, no general suite regression over5 points,
no newly introduced critical failure on the six predefined challenge turns,
required integrity/controls, and at least1GiB tested device headroom.

Generation length was fixed from validation reference lengths, not test
answers. Length-stopped outputs are failures. All150 test turns remain in
the denominator. The bootstrap resamples connected held-out tool/request
groups10,000 times. Seed variability and inference noise are kept separate.
Do not change thresholds or train again after inspecting these results and
call that the same final test.

Review the fixed validation sample in evidence/validation-reference-review.jsonl
before running this cell. Record any adjudication separately. The preparation
checks establish structure. Human review establishes whether a call matches
the request and whether a different call would also be valid.

B/M/Q also run all eight pinned Soup bundled general suites with native
Qwen prompt formatting and a256-token budget. These finite suites do not
establish universal safety. Raw predictions and exact scorer identities stay
in the evidence. Source-specific slices are unavailable in the four-file
export; single/multiple-call and length slices are reported instead.

In [12]:
stage('freeze')
for variant in ('B','M','Q'):
    stage('evaluate','--variant',variant,'--split','test','--general')
    stage('evaluate','--variant',variant,'--split','challenge')
soup('quantized-inference-bench','bench','infer','artifacts/merged-nf4',
     '--num-prompts','3','--max-tokens','64','--p50','--p95',heavy=True,timeout=900)

freeze started. Log: evidence/logs/freeze-1794297517735672079.log
freeze: exit=0; log=evidence/logs/freeze-1794297517735672079.log
evaluate started. Log: evidence/logs/evaluate-1794297517953844882.log
evaluate: exit=0; log=evidence/logs/evaluate-1794297517953844882.log
evaluate started. Log: evidence/logs/evaluate-1794298799145626107.log
evaluate: exit=0; log=evidence/logs/evaluate-1794298799145626107.log
evaluate started. Log: evidence/logs/evaluate-1794298828994340370.log
evaluate: exit=0; log=evidence/logs/evaluate-1794298828994340370.log
evaluate started. Log: evidence/logs/evaluate-1794299866831613520.log
evaluate: exit=0; log=evidence/logs/evaluate-1794299866831613520.log
evaluate started. Log: evidence/logs/evaluate-1794299893560445797.log
evaluate: exit=0; log=evidence/logs/evaluate-1794299893560445797.log
evaluate started. Log: evidence/logs/evaluate-1794301460849475655.log
evaluate: exit=0; log=evidence/logs/evaluate-1794301460849475655.log
quantized-inference-bench started. 

0

## Phase6 · Verdict, Soup decision replay and memory reconciliation

The project verdict is the conjunction of the frozen task, regression,
integrity, control and resource gates. It can legitimately be DON'T SHIP.
A failure should identify whether to repair data/training, re-export, or fix
an unresolved measurement; a pure export error does not justify retraining.

`soup ship --evidence` replays the actually measured B/M and B/Q scores.
Its provenance explicitly says **offline decision replay**, not independent
model inference. Soup quant-check is similarly reused on cached single-call
outputs; the full multi-call metric remains the stricter primary verifier.

This cell is safe to run after a failed earlier stage: absent required
reports yield DON'T SHIP with a missing-evidence list, never fabricated zeros.

In [13]:
stage('finalize')
show('verdict.json')

finalize started. Log: evidence/logs/finalize-1794301509105486253.log
finalize: exit=0; log=evidence/logs/finalize-1794301509105486253.log
{
  "verdict": "DON'T SHIP",
  "gates": {
    "required_stage_reports_pass": true,
    "policy_and_test_unchanged": true,
    "artifact_hashes_bound": true,
    "updated_and_active_adapter": true,
    "selected_frozen_weights_unchanged": true,
    "disabled_base_agreement": true,
    "merge_fidelity": false,
    "scorer_data_controls": true,
    "base_for_merge_control": true,
    "q0_for_q_control": true,
    "Soup_noop_adapter_control": true,
    "portable_M_Q": true,
    "real_saved_nf4": true,
    "no_new_critical_challenge_failures": true,
    "one_gib_tested_headroom": true,
    "three_seed_development_complete": true,
    "Soup_environment_bounds_ok": true,
    "Soup_fixture_coverage_resolved": true,
    "task_gain_ge_5pp": false,
    "task_gain_ci_positive": false,
    "quant_point_loss_le_2pp": false,
    "quant_ci_lower_ge_minus5pp": false

## Package the evidence and retained model artifacts

`evidence-and-code.zip` contains the standalone verifier, policy/configs,
exact inputs, raw timestamped logs/device traces, predictions, controls and
a two-page report draft. Model weights remain in the Kaggle Output artifact
folders to avoid duplicating them in that ZIP. Hashes and access paths are
indexed. Reflection and AI-use disclosure must describe your actual work;
fill `REFLECTION` from your observations and visually inspect the PDF before
submission. The notebook does not fabricate first-person experiences.

No files are published or sent to reviewers. Download the selected clean
adapter, merged-fp16 and merged-nf4 outputs and keep the failed-stage evidence.
Base weights can be reconstructed from the pinned model revision.

In [14]:
# Reflection affects the report only; it does not change performance policy or prior scores.
current = json.loads((RUN/'settings.json').read_text())
if current.get('reflection') != REFLECTION:
    # To keep completed-stage settings immutable, edit REFLECTION before the first run.
    raise ValueError('Use the report draft for final reflection edits; experiment settings are frozen.')
with (RUN/'evidence/nvidia-smi-final.raw.log').open('w') as handle:
    subprocess.run(['nvidia-smi'],stdout=handle,stderr=subprocess.STDOUT,check=False)
stage('package')
# Archive only after every child has exited and all raw logs are closed.
import zipfile
files = [p for p in (RUN/'evidence').rglob('*') if p.is_file() and not p.name.endswith('.tmp')]
files += [RUN/name for name in ('REPORT.md','report.pdf','REFLECTION_AND_AI_USE.md',
    'verdict.json','run_policy.json','settings.json','verify_artifacts.py','soup-data.yaml',
    'coarse-expectations.yaml') if (RUN/name).exists()]
files += list((RUN/'configs').glob('*.yaml')) + list((RUN/'data').glob('*.jsonl'))
index = {str(p.relative_to(RUN)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(set(files))}
(RUN/'evidence-index.json').write_text(json.dumps({'files':index,'models_excluded_from_zip':True,
    'model_access':'Retained under Kaggle Output/artifacts: selected adapter, merged-fp16, merged-nf4.',
    'standalone_verifier':'python verify_artifacts.py --root RUN evaluate --variant Q --split test --general'},indent=2)+'\n')
bundle = RUN/'evidence-and-code.zip'
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(files+[RUN/'evidence-index.json'])):
        archive.write(path,str(path.relative_to(RUN)))
with zipfile.ZipFile(bundle) as archive:
    assert archive.testzip() is None
    for name,expected in index.items():
        assert hashlib.sha256(archive.read(name)).hexdigest() == expected
from IPython.display import FileLink,display
display(FileLink(str(RUN/'report.pdf')))
display(FileLink(str(RUN/'evidence-and-code.zip')))
print('Retained artifacts:',RUN/'artifacts')
print('Standalone replay example:')
print(f'python {RUN / "verify_artifacts.py"} --root {RUN} evaluate --variant Q --split test --general')

package started. Log: evidence/logs/package-1794301527039629033.log
package: exit=0; log=evidence/logs/package-1794301527039629033.log


/kaggle/working/soup-experiment/20261110T044014005029Z/report.pdf

/kaggle/working/soup-experiment/20261110T044014005029Z/evidence-and-code.zip

Retained artifacts: /kaggle/working/soup-experiment/20261110T044014005029Z/artifacts
Standalone replay example:
python /kaggle/working/soup-experiment/20261110T044014005029Z/verify_artifacts.py --root /kaggle/working/soup-experiment/20261110T044014005029Z evaluate --variant Q --split test --general


In [17]:
import os
import sys
import subprocess
from pathlib import Path

# Configuration
SOURCE = Path("/kaggle/working/soup-experiment/20261110T044014005029Z")
MODEL_NAME = "soup-experiment"
VARIATION = "experiment-artifacts"

# Install KaggleHub if necessary
try:
    import kagglehub
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "kagglehub"
    ])
    import kagglehub

# Verify directory
assert SOURCE.is_dir(), f"Directory not found: {SOURCE}"

files = [p for p in SOURCE.rglob("*") if p.is_file()]
assert files, "Directory is empty!"

total_gb = sum(p.stat().st_size for p in files) / 1024**3
print(f"Files: {len(files)} | Size: {total_gb:.2f} GB")

# Authenticate
try:
    username = kagglehub.whoami()["username"]
except Exception:
    kagglehub.login()
    username = kagglehub.whoami()["username"]

handle = f"{username}/{MODEL_NAME}/pyTorch/{VARIATION}"

# Upload: an exception prevents proceeding on failure
print(f"Uploading to: {handle}")

kagglehub.model_upload(
    handle=handle,
    local_model_dir=str(SOURCE),
    version_notes="Soup experiment artifacts and checkpoints",
)

# Verify the published variation is accessible
uploaded_path = kagglehub.model_download(handle)

print("\nUPLOAD SUCCESSFUL")
print(f"Model: https://www.kaggle.com/models/{username}/{MODEL_NAME}")
print(f"Verified download path: {uploaded_path}")
print("Safe to stop the Kaggle session.")


Files: 408 | Size: 9.19 GB
Kaggle credentials successfully validated.
Uploading to: pernavjain/soup-experiment/pyTorch/experiment-artifacts
Uploading Model https://kaggle.com/models/pernavjain/soup-experiment/pyTorch/experiment-artifacts ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpl_zmqgek/archive.zip


Uploading: 100%|██████████| 9.87G/9.87G [02:27<00:00, 67.0MB/s]   

Upload successful: /tmp/tmpl_zmqgek/archive.zip (9GB)


Your model instance version has been created.
Files are being processed...
See at: https://kaggle.com/models/pernavjain/soup-experiment/pyTorch/experiment-artifacts
Mounting files to /kaggle/input/models/pernavjain/soup-experiment/pytorch/experiment-artifacts/1...

UPLOAD SUCCESSFUL
Model: https://www.kaggle.com/models/pernavjain/soup-experiment
Verified download path: /kaggle/input/models/pernavjain/soup-experiment/pytorch/experiment-artifacts/1
Safe to stop the Kaggle session.


## Source references and measured scope

- [Pinned Soup trainer](https://github.com/MakazhanAlpamys/Soup/blob/4cff43bf967e96a484381205e5125e2fc39227f1/src/soup_cli/trainer/sft.py)
- [Soup benchmark collector](https://github.com/MakazhanAlpamys/Soup/blob/4cff43bf967e96a484381205e5125e2fc39227f1/src/soup_cli/bench/collector.py)
- [Transformers5.20.2 checkpoint/evaluation arguments](https://huggingface.co/docs/transformers/v5.20.2/en/main_classes/trainer#transformers.TrainingArguments)
- [Soup function-call scorer](https://github.com/MakazhanAlpamys/Soup/blob/4cff43bf967e96a484381205e5125e2fc39227f1/src/soup_cli/eval/custom.py)
- [Soup general suites](https://github.com/MakazhanAlpamys/Soup/blob/4cff43bf967e96a484381205e5125e2fc39227f1/src/soup_cli/eval/gate_suites.py)
- [Soup merge](https://github.com/MakazhanAlpamys/Soup/blob/4cff43bf967e96a484381205e5125e2fc39227f1/src/soup_cli/commands/merge.py)
- [Pinned model configuration](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json)

Source compatibility and pure-helper tests can be checked before GPU
execution. Full notebook acceptance requires your actual T4 run. The earlier
four-update512-token smoke does not certify this1024-token full experiment.
The assignment's Colab environment and final human-edited submission remain
distinct from this Kaggle implementation. Use observed results in the report.